# GIBS Experiments — Agentic Trajectories

Graph Information Bottleneck for Stepwise Confidence (arXiv:2605.19228)

Adapted for airline agent tool-use trajectories.

**Train/Eval split (no leakage):**
- Train: 15 unlabeled tasks (300 runs) — MCS consensus masks as supervision
- Eval: 9 labeled tasks (180 runs) — AUROC against human annotations

**Models:**
1. GCN-GIBS: 2-layer GCN with heuristic edges (baseline)
2. TF-GIBS: Transformer encoder with learned attention (Agentic-GIBS)

In [149]:
!pip install -q transformers sentence-transformers scikit-learn tqdm torch

In [150]:
import sys, os, json, csv, math, random, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel
from torch.optim import Adam

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cuda


In [151]:
# ── Paths (same as main notebook) ────────────────────────────────────────────
DRIVE_BASE = '/home/svelag11/agentic'
sys.path.insert(0, DRIVE_BASE)

RESULTS   = DRIVE_BASE + '/results.json'
ANNOT_CSV = DRIVE_BASE + '/steps_to_annotate_all.csv'
CACHE_DIR = DRIVE_BASE + '/nli_cache'

os.environ['STEPWISE_NLI_CACHE'] = CACHE_DIR

# Cached file paths (shared with main notebook)
GIBS_DATA_PATH  = DRIVE_BASE + '/gibs_train_data.pkl'
GIBS_MODEL_PATH = DRIVE_BASE + '/gibs_model.pt'
TGIBS_MODEL_PATH = DRIVE_BASE + '/tgibs_model.pt'

print('Paths set')

Paths set


In [152]:
# ── Load data ─────────────────────────────────────────────────────────────────
data      = json.load(open(RESULTS))
info      = data['info']
tasks_map = {int(t['id']): t for t in data['tasks']}

sims_by_task = {}
for s in data['simulations']:
    tid = int(s['task_id'])
    sims_by_task.setdefault(tid, []).append(s)

print(f'Tasks: {len(tasks_map)}  |  Simulations: {len(data["simulations"])}')

Tasks: 50  |  Simulations: 1000


In [153]:
# ── Load annotations ──────────────────────────────────────────────────────────
with open(ANNOT_CSV) as f:
    annots = [
        r for r in csv.DictReader(f)
        if r.get('label') in ('correct', 'incorrect', 'tool_error')
    ]
print(f'Labeled steps: {len(annots)}')
print(f'  pos: {sum(1 for r in annots if r["label"] in ("incorrect","tool_error"))}')
print(f'  neg: {sum(1 for r in annots if r["label"] == "correct")}')

Labeled steps: 2857
  pos: 210
  neg: 2647


In [154]:
# ── Config ────────────────────────────────────────────────────────────────────
TASKS       = [1, 2, 9, 11, 12, 15, 18, 43, 47]
TRAIN_TASKS = [16,17,20,21,22,23,24,32,33,35,37,39,40,41,42]

# Expanded: all non-eval tasks with pass rate > 0% (~40 tasks, ~800 runs)
TRAIN_TASKS_EXPANDED = [
    0,3,4,5,6,8,10,13,14,
    16,17,19,20,21,22,23,24,
    25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,
    44,45,46,48,49
]

MASK_THRESH    = 0.7
NLI_DEP_THRESH = 0.3
USE_NLI_DEPS   = False

print(f'Config set  |  train={len(TRAIN_TASKS)} tasks  |  train_expanded={len(TRAIN_TASKS_EXPANDED)} tasks')


Config set  |  train=15 tasks  |  train_expanded=40 tasks


In [155]:
# ── MiniLM Encoder ────────────────────────────────────────────────────────────
EMB_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
EMBED_DIM = 384

emb_tok   = AutoTokenizer.from_pretrained(EMB_MODEL_NAME)
emb_model = AutoModel.from_pretrained(EMB_MODEL_NAME).to(DEVICE).eval()

@torch.no_grad()
def encode_texts(texts, batch_size=64):
    if not texts: return torch.zeros(0, EMBED_DIM)
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = emb_tok(batch, padding=True, truncation=True,
                        max_length=128, return_tensors='pt').to(DEVICE)
        m     = enc['attention_mask'].unsqueeze(-1).float()
        h     = emb_model(**enc).last_hidden_state
        vecs.append((h * m).sum(1) / m.sum(1))
    return torch.cat(vecs).cpu()

print('MiniLM encoder ready')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

MiniLM encoder ready


In [156]:
# ── gibs/graph/step_graph.py (inlined) ───────────────────────────────────────
import json, re
from typing import Any

ENUM_VALUES = {"yes", "no", "true", "false", "economy", "business", "basic_economy", "one_way", "round_trip"}
VALUE_TOKEN = re.compile(
    r"\d{4}-\d{2}-\d{2}"
    r"|\b[A-Z0-9]{6}\b"
    r"|\b[a-z]+_[a-z]+_\d+\b"
    r"|\b(?:gift_card|credit_card|certificate)_\d+\b"
    r"|\bHAT\d{3}\b"
)
FREE_TEXT_MIN_LEN = 40
ISO_DATE = re.compile(r"^\d{4}-\d{2}-\d{2}$")
CONTROL_TOKEN = re.compile(r"^###([A-Z_\-]+)###$")
CONTROL_TEXT = {
    "STOP": "The user ended the conversation.",
    "TRANSFER": "The user was transferred to a human agent.",
    "OUT-OF-SCOPE": "The user said the request is out of scope.",
}
UNICODE_FIXES = {"\u2010": "-", "\u2011": "-", "\u2012": "-", "\u2013": "-", "\u2014": "-",
                 "\u00a0": " ", "\u202f": " ", "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"'}

def normalize_text(text):
    for k, v in UNICODE_FIXES.items():
        text = text.replace(k, v)
    text = text.replace("**", "").replace("__", "")
    return re.sub(r"\s+", " ", text).strip()

def _flatten(value, prefix=""):
    if isinstance(value, dict):
        out = []
        for k, v in value.items():
            out += _flatten(v, f"{prefix}.{k}" if prefix else str(k))
        return out
    if isinstance(value, list):
        if not value:
            return [(prefix, "[]")]
        out = []
        for i, v in enumerate(value):
            out += _flatten(v, f"{prefix}[{i}]")
        return out
    return [(prefix, value)]

def _scalar(v):
    return "null" if v is None else (str(v).lower() if isinstance(v, bool) else str(v))

def verbalize_tool_call(name, arguments):
    pairs = _flatten(arguments or {})
    if not pairs:
        return f"Call {name} with no arguments."
    return f"Call {name} with " + "; ".join(f"{k} = {normalize_text(_scalar(v))}" for k, v in pairs) + "."

def verbalize_observation(content, error):
    if error:
        return "Tool error: " + normalize_text(content)
    try:
        parsed = json.loads(content)
    except (json.JSONDecodeError, TypeError):
        return "Tool result: " + normalize_text(str(content))
    if isinstance(parsed, (dict, list)):
        return "Tool result: " + "; ".join(f"{k}: {normalize_text(_scalar(v))}" for k, v in _flatten(parsed)) + "."
    return "Tool result: " + normalize_text(_scalar(parsed))

def verbalize_utterance(content):
    text = (content or "").strip()
    m = CONTROL_TOKEN.match(text)
    if m:
        return CONTROL_TEXT.get(m.group(1), f"The user sent the control signal {m.group(1)}.")
    return normalize_text(text)

def _reasoning(msg):
    try:
        m = msg["raw_data"]["choices"][0]["message"]
    except (KeyError, IndexError, TypeError):
        return None
    return m.get("reasoning") or m.get("reasoning_content") or None

def _leaves(v):
    if isinstance(v, dict):
        for x in v.values(): yield from _leaves(x)
    elif isinstance(v, list):
        for x in v: yield from _leaves(x)
    else:
        yield v

def dependency_values(arguments):
    out = []
    for v in _leaves(arguments or {}):
        if not isinstance(v, str):
            continue
        if len(v) >= FREE_TEXT_MIN_LEN:
            cands = VALUE_TOKEN.findall(v)
        elif len(v) < 3 or v.lower() in ENUM_VALUES:
            cands = []
        else:
            cands = [v]
        for c in cands:
            if c not in out:
                out.append(c)
    return out

def value_occurs(value, text):
    left = r"(?<![A-Za-z0-9])"
    right = r"(?![A-Za-z0-9])" if not ISO_DATE.match(value) else r"(?![A-SU-Za-z0-9])"
    return re.search(left + re.escape(value) + right, text) is not None

def _visible_text(node):
    if node["type"] == "UserTurn":
        return node.get("content") or ""
    if node.get("kind") == "tool_call":
        return node["observation"]["content_raw"] or ""
    return node.get("content") or ""

def build_step_graph(sim, info=None, task=None, source_file=None):
    msgs = sim.get("messages") or []
    nodes = []
    step_index = 0
    i = 0
    while i < len(msgs):
        m = msgs[i]
        role = m["role"]
        if role == "user":
            nodes.append({
                "id": f"u{i}", "type": "UserTurn", "scorable": False,
                "raw_message_indices": [i], "turn_idx": m.get("turn_idx"), "timestamp": m.get("timestamp"),
                "content": m.get("content"),
                "op_text": verbalize_utterance(m.get("content")),
                "result_text": verbalize_utterance(m.get("content")),
            })
        elif role == "assistant":
            node = {
                "id": f"s{i}", "type": "AgentStep", "scorable": True, "step_index": step_index,
                "raw_message_indices": [i], "turn_idx": m.get("turn_idx"), "timestamp": m.get("timestamp"),
                "reasoning": _reasoning(m), "content": m.get("content"),
            }
            calls = m.get("tool_calls") or []
            if len(calls) > 1:
                raise ValueError(f"message {i}: parallel tool calls not supported")
            if calls:
                tc = calls[0]
                if i + 1 >= len(msgs) or msgs[i+1]["role"] != "tool" or msgs[i+1].get("id") != tc["id"]:
                    raise ValueError(f"message {i}: tool call {tc['id']} has no immediately following observation")
                obs = msgs[i+1]
                try:
                    parsed = json.loads(obs["content"])
                except (json.JSONDecodeError, TypeError):
                    parsed = None
                node.update({
                    "kind": "tool_call",
                    "tool_call": {"id": tc["id"], "name": tc["name"], "arguments": tc.get("arguments") or {}},
                    "observation": {"tool_call_id": obs["id"], "error": bool(obs.get("error")),
                                    "raw_message_index": i+1, "timestamp": obs.get("timestamp"),
                                    "content_raw": obs["content"], "content_parsed": parsed},
                    "op_text": verbalize_tool_call(tc["name"], tc.get("arguments") or {}),
                    "result_text": verbalize_observation(obs["content"], bool(obs.get("error"))),
                })
                node["raw_message_indices"].append(i+1)
                i += 1
            else:
                node.update({"kind": "message",
                             "op_text": verbalize_utterance(m.get("content")),
                             "result_text": verbalize_utterance(m.get("content"))})
            nodes.append(node)
            step_index += 1
        else:
            raise ValueError(f"message {i}: unpaired role {role!r}")
        i += 1

    for k, n in enumerate(nodes):
        if n["type"] != "AgentStep": continue
        if n["kind"] == "tool_call":
            n["response_node"] = n["id"]
        else:
            nxt = nodes[k+1] if k+1 < len(nodes) else None
            n["response_node"] = nxt["id"] if nxt and nxt["type"] == "UserTurn" else None

    edges = [{"id": f"t{k}", "type": "temporal", "source": a["id"], "target": b["id"]}
             for k, (a, b) in enumerate(zip(nodes, nodes[1:]))]
    dep_count = 0
    for k, n in enumerate(nodes):
        if n.get("kind") != "tool_call": continue
        grouped = {}
        for value in dependency_values(n["tool_call"]["arguments"]):
            hits = [p["id"] for p in nodes[:k] if value_occurs(value, _visible_text(p))]
            if not hits: continue
            src = hits[0]
            g = grouped.setdefault(src, {"values": [], "alternative_sources": {}})
            g["values"].append(value)
            if hits[1:]: g["alternative_sources"][value] = hits[1:]
        for src, g in grouped.items():
            edges.append({"id": f"d{dep_count}", "type": "data_dependency", "source": src, "target": n["id"],
                          "values": g["values"], "alternative_sources": g["alternative_sources"],
                          "rule": "earliest prior node whose raw visible text contains the value"})
            dep_count += 1

    info = info or {}
    ri = sim.get("reward_info") or {}
    reward = ri.get("reward")
    metadata = {
        "source_file": source_file, "simulation_id": sim.get("id"),
        "task_id": str(sim.get("task_id")), "trial": sim.get("trial"),
        "reward": reward, "success": None if reward is None else reward >= 0.5,
    }
    return {"schema_version": "0.2",
            "graph_id": f"task{metadata['task_id']}_{str(sim.get('id'))[:8]}",
            "metadata": metadata, "nodes": nodes, "edges": edges}

print('step_graph ready')

step_graph ready


In [157]:
# ── gibs/mcs/nli.py (inlined, essential parts only) ──────────────────────────
import hashlib, os
from pathlib import Path

DEFAULT_CACHE_DIR = Path(os.environ.get("STEPWISE_NLI_CACHE", Path.home() / ".cache" / "stepwise_nli"))

def symmetric_input_ids(tokenizer, a, b, max_length):
    full = tokenizer(f"{a} [SEP] {b}", truncation=False)["input_ids"]
    cls, sep = tokenizer.cls_token_id, tokenizer.sep_token_id
    mid = full.index(sep, 1)
    end = mid
    while full[end+1] == sep and end+1 < len(full)-1:
        end += 1
    sep_block = full[mid:end+1]
    ids_a, ids_b = full[1:mid], full[end+1:-1]
    budget = max_length - 2 - len(sep_block)
    la, lb = len(ids_a), len(ids_b)
    if la + lb <= budget:
        ka, kb = la, lb
    else:
        half = budget // 2
        if la <= half:   ka, kb = la, budget - la
        elif lb <= half: ka, kb = budget - lb, lb
        else:            ka, kb = half, budget - half
    return [cls] + ids_a[:ka] + sep_block + ids_b[:kb] + [sep]


class DebertaEntailment:
    def __init__(self, model_name="microsoft/deberta-large-mnli", device=None,
                 batch_size=16, cache_dir=DEFAULT_CACHE_DIR, overlength="symmetric"):
        import torch
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
        self.overlength = overlength
        self.truncated = set()
        self.n_forward = 0
        self.n_requested = 0
        self.torch = torch
        self.model_name = model_name
        self.batch_size = batch_size
        if device is None:
            if torch.cuda.is_available():    self.device = "cuda"
            elif torch.backends.mps.is_available(): self.device = "mps"
            else:                            self.device = "cpu"
        else:
            self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name, torch_dtype=torch.float32)
        self.model.to(self.device).eval()
        labels = {v.lower(): int(k) for k, v in self.model.config.id2label.items()}
        self.entail_index = labels["entailment"]
        try:
            from huggingface_hub import snapshot_download
            self.revision = Path(snapshot_download(model_name, local_files_only=True)).name
        except Exception:
            self.revision = getattr(self.model.config, "_commit_hash", None) or "unknown"
        self.max_length = min(self.tokenizer.model_max_length, self.model.config.max_position_embeddings)
        self.cache = {}
        self.cache_file = None
        if cache_dir is not None:
            cache_dir = Path(cache_dir)
            cache_dir.mkdir(parents=True, exist_ok=True)
            tag = f"{model_name.replace('/', '__')}__{self.revision[:12]}__code_sep_string"
            self.cache_file = cache_dir / f"{tag}.jsonl"
            if self.cache_file.exists():
                with open(self.cache_file) as f:
                    for line in f:
                        rec = json.loads(line)
                        self.cache[rec["k"]] = rec["p"]
        print(f"{model_name} | {self.device} | cache: {len(self.cache)} entries")

    def _key(self, a, b, truncated=False):
        tag = "" if not truncated else chr(0) + "symmetric"
        return hashlib.sha256(f"{a}\x00{b}{tag}".encode()).hexdigest()

    def token_length(self, a, b):
        return len(self.tokenizer(f"{a} [SEP] {b}", truncation=False)["input_ids"])

    def encode(self, pairs, pad=True, truncate=False):
        if truncate and self.overlength == "symmetric":
            rows = [symmetric_input_ids(self.tokenizer, a, b, self.max_length) for a, b in pairs]
            width = max(len(r) for r in rows)
            pad_id = self.tokenizer.pad_token_id
            ids  = self.torch.tensor([r + [pad_id]*(width-len(r)) for r in rows])
            mask = self.torch.tensor([[1]*len(r) + [0]*(width-len(r)) for r in rows])
            return {"input_ids": ids, "attention_mask": mask}
        kw = {"return_tensors": "pt", "padding": True} if pad else {}
        kw.update({"truncation": True, "max_length": self.max_length} if truncate else {"truncation": False})
        return self.tokenizer([f"{a} [SEP] {b}" for a, b in pairs], **kw)

    def entail_batch(self, pairs):
        unique = list(dict.fromkeys(pairs))
        self.n_requested += len(unique)
        long = {(a, b) for a, b in unique if self.token_length(a, b) > self.max_length}
        self.truncated |= long
        new = []
        with self.torch.no_grad():
            for truncate in (False, True):
                todo = [p for p in unique if (p in long)==truncate and self._key(*p, truncate) not in self.cache]
                for i in range(0, len(todo), self.batch_size):
                    chunk = todo[i:i+self.batch_size]
                    enc = {k: v.to(self.device) for k, v in self.encode(chunk, truncate=truncate).items()}
                    probs = self.torch.softmax(self.model(**enc).logits.float(), dim=-1)[:, self.entail_index]
                    self.n_forward += len(chunk)
                    for (a, b), p in zip(chunk, probs.cpu().tolist()):
                        self.cache[self._key(a, b, truncate)] = p
                        new.append({"k": self._key(a, b, truncate), "p": p})
        if new and self.cache_file is not None:
            with open(self.cache_file, "a") as f:
                for rec in new:
                    f.write(json.dumps(rec) + "\n")
        return [self.cache[self._key(a, b, (a,b) in long)] for a, b in pairs]

    def describe(self):
        return {"model": self.model_name, "device": self.device}


class Similarity:
    def __init__(self, model, direction="max_bidirectional"):
        self.model = model
        self.direction = direction
        self._memo = {}

    def prefetch(self, pairs):
        pairs = [p for p in dict.fromkeys(pairs) if p not in self._memo]
        if not pairs: return
        fwd = self.model.entail_batch(pairs)
        bwd = self.model.entail_batch([(b, a) for a, b in pairs]) if self.direction == "max_bidirectional" else [None]*len(pairs)
        for p, f, b in zip(pairs, fwd, bwd):
            self._memo[p] = (f, b)

    def __call__(self, a, b):
        if (a, b) not in self._memo:
            self.prefetch([(a, b)])
        f, b_ = self._memo[(a, b)]
        return f if b_ is None else max(f, b_)

print('nli ready')

nli ready


In [158]:
import os
CACHE_DIR = DRIVE_BASE + '/nli_cache'
os.environ['STEPWISE_NLI_CACHE'] = CACHE_DIR   # kept for reference / status cell
nli = DebertaEntailment(overlength='symmetric', batch_size=32, cache_dir=CACHE_DIR)
if nli.device == 'cuda':
    nli.model = nli.model.half()
    print('fp16 enabled')
sim = Similarity(nli)

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-large-mnli
Key    | Status     |  | 
-------+------------+--+-
config | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


microsoft/deberta-large-mnli | cuda | cache: 1072981 entries
fp16 enabled


In [159]:
# ── Dependency utilities ──────────────────────────────────────────────────────
NLI_DEP_THRESH = 0.3
USE_NLI_DEPS   = False

def get_deps(sg):
    if USE_NLI_DEPS:
        return _assign_nli_deps(sg)
    dep_map = {e["target"]: e["source"] for e in sg["edges"] if e["type"] == "data_dependency"}
    return {n["id"]: dep_map.get(n["id"]) for n in sg["nodes"] if n["type"] == "AgentStep"}

def _assign_nli_deps(sg):
    nodes = sg["nodes"]
    pairs = []
    for i, n in enumerate(nodes):
        if n["type"] != "AgentStep": continue
        srcs = nodes[:i] if n["kind"] == "tool_call" else [p for p in nodes[:i] if p["type"] == "UserTurn"]
        for s in srcs:
            pairs.append((s["result_text"], n["op_text"]))
    sim.prefetch(pairs)
    deps = {}
    for i, n in enumerate(nodes):
        if n["type"] != "AgentStep": continue
        srcs = nodes[:i] if n["kind"] == "tool_call" else [p for p in nodes[:i] if p["type"] == "UserTurn"]
        if not srcs: deps[n["id"]] = None; continue
        scores = [sim(s["result_text"], n["op_text"]) for s in srcs]
        best   = int(__import__("numpy").argmax(scores))
        deps[n["id"]] = srcs[best]["id"] if scores[best] >= NLI_DEP_THRESH else None
    return deps

def build_adj_norm(n, edge_pairs):
    import torch
    A = torch.zeros(n, n)
    for i, j in edge_pairs:
        A[i, j] = 1.0; A[j, i] = 1.0
    A = A + torch.eye(n)
    D = A.sum(1)
    D_inv_sqrt = torch.diag(D.pow(-0.5))
    return D_inv_sqrt @ A @ D_inv_sqrt

print("get_deps + build_adj_norm ready")

def get_last_user_text(nodes, before_idx):
    for n in reversed(nodes[:before_idx]):
        if n['type'] == 'UserTurn':
            return n.get('op_text') or ''
    return ''



get_deps + build_adj_norm ready


In [160]:
# ── Build training data (with cache) ─────────────────────────────────────────
def build_gibs_data(tid):
    task      = tasks_map[tid]
    all_sims  = sims_by_task.get(tid, [])
    pass_sims = [s for s in all_sims if (s.get('reward_info') or {}).get('reward', 0) >= 0.5]
    if len(pass_sims) < 2: return []

    ref_texts = []
    for ps in pass_sims:
        try:
            sg = build_step_graph(ps, info, task)
            ref_texts += [n['op_text'] for n in sg['nodes']
                          if n['type'] == 'AgentStep' and n.get('op_text')]
        except Exception:
            continue
    if not ref_texts: return []

    results = []
    for run in all_sims:
        try:
            sg   = build_step_graph(run, info, task)
            deps = get_deps(sg)
        except Exception:
            continue

        agent_nodes = [(i, n) for i, n in enumerate(sg['nodes']) if n['type'] == 'AgentStep']
        if not agent_nodes: continue

        q_texts = [n['op_text'] or '' for _, n in agent_nodes]

        # Prefetch NLI pairs
        pairs = [(qt, rt) for qt in q_texts for rt in ref_texts]
        sim.prefetch(pairs + [(b, a) for a, b in pairs])

        # Consensus mask
        mask = [1.0 if max(sim(qt, rt) for rt in ref_texts) >= MASK_THRESH else 0.0
                for qt in q_texts]

        # Node features
        x = encode_texts(q_texts)

        # Adjacency
        id2pos = {n['id']: pos for pos, (_, n) in enumerate(agent_nodes)}
        ep     = [(pos, pos+1) for pos in range(len(agent_nodes)-1)]
        for _, n in agent_nodes:
            ds = deps.get(n['id'])
            if ds and ds in id2pos and n['id'] in id2pos:
                ep.append((id2pos[ds], id2pos[n['id']]))
        adj = build_adj_norm(len(agent_nodes), ep)

        results.append((x, adj, torch.tensor(mask)))
    return results

if os.path.exists(GIBS_DATA_PATH):
    with open(GIBS_DATA_PATH, 'rb') as f:
        all_train_data = pickle.load(f)
    print(f'Loaded cached training data: {len(all_train_data)} runs')
else:
    all_train_data = []
    for tid in tqdm(TRAIN_TASKS, desc='Building training data'):
        all_train_data.extend(build_gibs_data(tid))
    with open(GIBS_DATA_PATH, 'wb') as f:
        pickle.dump(all_train_data, f)
    print(f'Built and cached: {len(all_train_data)} runs')

total_nodes = sum(x.shape[0] for x, _, _ in all_train_data)
pos_ratio   = sum(m.mean().item() for _, _, m in all_train_data) / len(all_train_data)
print(f'Training graphs : {len(all_train_data)}')
print(f'Total nodes     : {total_nodes}')
print(f'Avg consensus % : {pos_ratio:.2f}')

Loaded cached training data: 300 runs
Training graphs : 300
Total nodes     : 5160
Avg consensus % : 0.86


## IB Loss (shared)

`H(p) + λ·CE(p, mask)` — entropy for sparsity + weighted CE for consensus alignment.

Weighted CE penalises mask=0 (incorrect) steps 5x more to counter 86% positive rate.

In [161]:
# ── IB Loss (used by both models) ─────────────────────────────────────────────
def ib_loss(p, mask, lam=1.0, pos_weight=5.0):
    eps = 1e-7
    H   = -(p*(p+eps).log() + (1-p)*(1-p+eps).log()).mean()
    w   = torch.where(mask > 0.5,
                      torch.ones_like(mask),
                      torch.full_like(mask, pos_weight))
    ce  = F.binary_cross_entropy(p, mask, weight=w)
    return H + lam * ce

print('IB loss ready')

IB loss ready


## 1. GCN-GIBS (Baseline)

In [162]:
class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim, bias=False)
    def forward(self, x, adj):
        return F.relu(self.lin(adj.to(x.device) @ x))

class GIBSModel(nn.Module):
    def __init__(self, embed_dim=384, hidden=64):
        super().__init__()
        self.gcn1 = GCNLayer(embed_dim, hidden)
        self.gcn2 = GCNLayer(hidden, hidden)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x, adj):
        h = self.gcn1(x, adj)
        h = self.gcn2(h, adj)
        return torch.sigmoid(self.head(h)).squeeze(-1)

gibs_model = GIBSModel().to(DEVICE)
print(f'GCN-GIBS params: {sum(p.numel() for p in gibs_model.parameters()):,}')

GCN-GIBS params: 28,737


In [163]:
if os.path.exists(GIBS_MODEL_PATH):
    gibs_model.load_state_dict(torch.load(GIBS_MODEL_PATH, map_location=DEVICE))
    gibs_model.eval()
    print(f'Loaded saved model from {GIBS_MODEL_PATH}')
else:
    optimizer = Adam(gibs_model.parameters(), lr=1e-3, weight_decay=1e-4)
    gibs_model.train()
    for epoch in range(1, 31):
        total_loss = 0.0
        random.shuffle(all_train_data)
        for x, adj, mask in all_train_data:
            x, mask = x.to(DEVICE), mask.to(DEVICE)
            optimizer.zero_grad()
            p    = gibs_model(x, adj)
            loss = ib_loss(p, mask)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if epoch % 5 == 0:
            print(f'Epoch {epoch:3d}/30  loss={total_loss/len(all_train_data):.4f}')
    torch.save(gibs_model.state_dict(), GIBS_MODEL_PATH)
    print(f'Training complete. Model saved to {GIBS_MODEL_PATH}')

Loaded saved model from /home/svelag11/agentic/gibs_model.pt


In [164]:
gibs_model.eval()
all_consensus_gibs = {}

with tqdm(total=len(TASKS), desc='GCN-GIBS inference', unit='task') as pbar:
    for tid in TASKS:
        task     = tasks_map[tid]
        all_sims = sims_by_task.get(tid, [])
        for run in all_sims:
            try:
                sg = build_step_graph(run, info, task)
            except Exception:
                continue
            agent_nodes_with_idx = [(i, n) for i, n in enumerate(sg['nodes'])
                                    if n['type'] == 'AgentStep']
            if not agent_nodes_with_idx: continue
            texts  = [n.get('op_text') or '' for _, n in agent_nodes_with_idx]
            x      = encode_texts(texts).to(DEVICE)
            deps   = get_deps(sg)
            id2pos = {n['id']: pos for pos, (_, n) in enumerate(agent_nodes_with_idx)}
            ep     = [(pos, pos+1) for pos in range(len(agent_nodes_with_idx)-1)]
            for _, n in agent_nodes_with_idx:
                ds = deps.get(n['id'])
                if ds and ds in id2pos and n['id'] in id2pos:
                    ep.append((id2pos[ds], id2pos[n['id']]))
            adj = build_adj_norm(len(agent_nodes_with_idx), ep)
            with torch.no_grad():
                p = gibs_model(x, adj)
            all_consensus_gibs[run['id']] = {
                step_idx: {'m_gibs': float(p[pos])}
                for pos, (step_idx, _) in enumerate(agent_nodes_with_idx)
            }
        pbar.update(1)

print(f'Runs scored (GCN-GIBS): {len(all_consensus_gibs)}')

y_true, s_gibs = [], []
for row in annots:
    sd = all_consensus_gibs.get(row['run_id'], {}).get(int(row['step_idx']))
    if sd is None: continue
    y_true.append(1 if row['label'] in ('incorrect', 'tool_error') else 0)
    s_gibs.append(1 - sd['m_gibs'])

print(f'Steps matched  : {len(y_true)}  (pos={sum(y_true)}, neg={len(y_true)-sum(y_true)})')
print(f'AUROC GCN-GIBS : {roc_auc_score(y_true, s_gibs):.4f}')

GCN-GIBS inference: 100%|██████████| 9/9 [00:02<00:00,  3.46task/s]

Runs scored (GCN-GIBS): 180
Steps matched  : 2857  (pos=210, neg=2647)
AUROC GCN-GIBS : 0.7615


## 1b. GCN + NLI Features

Same GCN architecture but node features extended to 386-dim:
`[MiniLM(384) | NLI(op, last_user_msg) | NLI(op, policy)]`

Reuses `all_tgibs_data` (cached from TF-v2 cell) — no new data build needed.
Uses soft MCS masks (continuous NLI score, same as TF-v2).

In [165]:
# ── GCN + NLI Features (386-dim) ─────────────────────────────────────────────
GCN_NLI_MODEL_PATH = DRIVE_BASE + '/gibs_gcn_nli_model.pt'

# Reuse GCNLayer (already defined above) — just change in_dim to 386
gcn_nli_model = GIBSModel(embed_dim=386).to(DEVICE)
print(f'GCN+NLI params: {sum(p.numel() for p in gcn_nli_model.parameters()):,}')


GCN+NLI params: 28,865


In [166]:
# ── GCN + NLI Training ────────────────────────────────────────────────────────
# Load TF-v2 data (386-dim) if not already in memory
if 'all_tgibs_data' not in dir():
    if os.path.exists(DRIVE_BASE + '/tgibs_train_data.pkl'):
        with open(DRIVE_BASE + '/tgibs_train_data.pkl', 'rb') as f:
            all_tgibs_data = pickle.load(f)
        print(f'Loaded {len(all_tgibs_data)} runs from cache')
    else:
        raise RuntimeError('Run cell-tfv2-data first to build all_tgibs_data')

# To retrain: import os; os.remove(GCN_NLI_MODEL_PATH)

if os.path.exists(GCN_NLI_MODEL_PATH):
    gcn_nli_model.load_state_dict(torch.load(GCN_NLI_MODEL_PATH, map_location=DEVICE))
    gcn_nli_model.eval()
    print(f'Loaded saved model from {GCN_NLI_MODEL_PATH}')
else:
    optimizer = torch.optim.AdamW(gcn_nli_model.parameters(), lr=3e-4, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
    gcn_nli_model.train()
    for epoch in range(1, 51):
        total_loss = 0.0
        random.shuffle(all_tgibs_data)
        for x, adj, mask in all_tgibs_data:
            x, adj, mask = x.to(DEVICE), adj.to(DEVICE), mask.to(DEVICE)
            optimizer.zero_grad()
            p    = gcn_nli_model(x, adj)
            loss = ib_loss(p, mask, lam=2.0)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(gcn_nli_model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if epoch % 10 == 0:
            print(f'Epoch {epoch:3d}/50  loss={total_loss/len(all_tgibs_data):.4f}')
    torch.save(gcn_nli_model.state_dict(), GCN_NLI_MODEL_PATH)
    print(f'Training complete. Saved to {GCN_NLI_MODEL_PATH}')


Loaded saved model from /home/svelag11/agentic/gibs_gcn_nli_model.pt


In [167]:
# ── GCN + NLI Inference + AUROC ─────────────────────────────────────────────
policy_text = (info.get('environment_info') or {}).get('policy', '')
gcn_nli_model.eval()
all_consensus_gcn_nli = {}

with tqdm(total=len(TASKS), desc='GCN+NLI inference', unit='task') as pbar:
    for tid in TASKS:
        task     = tasks_map[tid]
        all_sims = sims_by_task.get(tid, [])
        for run in all_sims:
            try:
                sg = build_step_graph(run, info, task)
            except Exception:
                continue
            nodes = sg['nodes']
            agent_nodes_with_idx = [(i, n) for i, n in enumerate(nodes)
                                    if n['type'] == 'AgentStep']
            if not agent_nodes_with_idx: continue

            q_texts   = [n.get('op_text') or '' for _, n in agent_nodes_with_idx]
            user_msgs = [get_last_user_text(nodes, i) for i, _ in agent_nodes_with_idx]
            user_align   = [sim(qt, um) if um else 0.5 for qt, um in zip(q_texts, user_msgs)]
            policy_score = [sim(qt, policy_text) if policy_text else 0.5 for qt in q_texts]

            x_base  = encode_texts(q_texts)
            x_extra = torch.tensor([[ua, ps] for ua, ps in zip(user_align, policy_score)])
            x_aug   = torch.cat([x_base, x_extra], dim=1).to(DEVICE)

            id2pos  = {n['id']: pos for pos, (_, n) in enumerate(agent_nodes_with_idx)}
            dep_map = {e['target']: e['source'] for e in sg['edges'] if e['type'] == 'data_dependency'}
            n_steps = len(agent_nodes_with_idx)
            ep = [(pos, pos+1) for pos in range(n_steps-1)]
            for _, n in agent_nodes_with_idx:
                src = dep_map.get(n['id'])
                if src and src in id2pos and n['id'] in id2pos:
                    ep.append((id2pos[src], id2pos[n['id']]))
            adj = build_adj_norm(n_steps, ep).to(DEVICE)

            with torch.no_grad():
                p = gcn_nli_model(x_aug, adj)

            orig_indices = [i for i, _ in agent_nodes_with_idx]
            all_consensus_gcn_nli[run['id']] = {
                orig_idx: {'m': float(p[pos])}
                for pos, orig_idx in enumerate(orig_indices)
            }
        pbar.update(1)

print(f'Runs scored: {len(all_consensus_gcn_nli)}')

y_true, s_gcn_nli = [], []
for row in annots:
    sd = all_consensus_gcn_nli.get(row['run_id'], {}).get(int(row['step_idx']))
    if sd is None: continue
    y_true.append(1 if row['label'] in ('incorrect', 'tool_error') else 0)
    s_gcn_nli.append(1 - sd['m'])

print(f'Steps matched    : {len(y_true)}  (pos={sum(y_true)}, neg={len(y_true)-sum(y_true)})')
print()
print(f'AUROC GCN+NLI    : {roc_auc_score(y_true, s_gcn_nli):.4f}  <- 386-dim + graph structure')
print(f'--- known results ---')
print(f'NA-MCS           :  0.8101')
print(f'GCN-GIBS         :  0.7615  (384-dim, no NLI features)')
print(f'TF-GIBS v2       :  0.6943  (386-dim, no graph structure)')


GCN+NLI inference: 100%|██████████| 9/9 [00:12<00:00,  1.37s/task]

Runs scored: 180
Steps matched    : 2857  (pos=210, neg=2647)

AUROC GCN+NLI    : 0.8718  <- 386-dim + graph structure
--- known results ---
NA-MCS           :  0.8101
GCN-GIBS         :  0.7615  (384-dim, no NLI features)
TF-GIBS v2       :  0.6943  (386-dim, no graph structure)


## 2. Transformer-GIBS (Agentic-GIBS)

Replaces GCN + heuristic edges with Transformer encoder + causal self-attention.
No graph structure needed — attention learns step dependencies from data.

In [168]:
class TransformerGIBS(nn.Module):
    def __init__(self, embed_dim=384, d_model=128, nhead=4, num_layers=2, dropout=0.2):
        super().__init__()
        self.proj    = nn.Linear(embed_dim, d_model)
        enc_layer    = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dropout=dropout,
            batch_first=True, dim_feedforward=d_model*4
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x):
        n = x.size(0)
        x = self.proj(x)
        causal_mask = torch.triu(torch.ones(n, n, device=x.device), diagonal=1).bool()
        out = self.transformer(x.unsqueeze(0), mask=causal_mask, is_causal=True)
        return torch.sigmoid(self.head(out.squeeze(0))).squeeze(-1)

tgibs_model = TransformerGIBS().to(DEVICE)
print(f'TF-GIBS params: {sum(p.numel() for p in tgibs_model.parameters()):,}')

TF-GIBS params: 445,953


In [169]:
# To retrain: delete TGIBS_MODEL_PATH first
# import os; os.remove(TGIBS_MODEL_PATH)

if os.path.exists(TGIBS_MODEL_PATH):
    tgibs_model.load_state_dict(torch.load(TGIBS_MODEL_PATH, map_location=DEVICE))
    tgibs_model.eval()
    print(f'Loaded saved model from {TGIBS_MODEL_PATH}')
else:
    optimizer = Adam(tgibs_model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
    tgibs_model.train()
    for epoch in range(1, 31):
        total_loss = 0.0
        random.shuffle(all_train_data)
        for x, _, mask in all_train_data:  # adj not needed
            x, mask = x.to(DEVICE), mask.to(DEVICE)
            optimizer.zero_grad()
            p    = tgibs_model(x)
            loss = ib_loss(p, mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(tgibs_model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if epoch % 5 == 0:
            print(f'Epoch {epoch:3d}/30  loss={total_loss/len(all_train_data):.4f}')
    torch.save(tgibs_model.state_dict(), TGIBS_MODEL_PATH)
    print(f'Training complete. Model saved to {TGIBS_MODEL_PATH}')

Loaded saved model from /home/svelag11/agentic/tgibs_model.pt


In [170]:
tgibs_model.eval()
all_consensus_tgibs = {}

with tqdm(total=len(TASKS), desc='TF-GIBS inference', unit='task') as pbar:
    for tid in TASKS:
        task     = tasks_map[tid]
        all_sims = sims_by_task.get(tid, [])
        for run in all_sims:
            try:
                sg = build_step_graph(run, info, task)
            except Exception:
                continue
            agent_nodes_with_idx = [(i, n) for i, n in enumerate(sg['nodes'])
                                    if n['type'] == 'AgentStep']
            if not agent_nodes_with_idx: continue
            texts = [n.get('op_text') or '' for _, n in agent_nodes_with_idx]
            x     = encode_texts(texts).to(DEVICE)
            with torch.no_grad():
                p = tgibs_model(x)
            all_consensus_tgibs[run['id']] = {
                step_idx: {'m_tgibs': float(p[pos])}
                for pos, (step_idx, _) in enumerate(agent_nodes_with_idx)
            }
        pbar.update(1)

print(f'Runs scored (TF-GIBS): {len(all_consensus_tgibs)}')

y_true, s_gcn, s_tf = [], [], []
for row in annots:
    sd_g  = all_consensus_gibs.get(row['run_id'],  {}).get(int(row['step_idx']))
    sd_tf = all_consensus_tgibs.get(row['run_id'], {}).get(int(row['step_idx']))
    if sd_g is None or sd_tf is None: continue
    y_true.append(1 if row['label'] in ('incorrect', 'tool_error') else 0)
    s_gcn.append(1 - sd_g['m_gibs'])
    s_tf.append(1 - sd_tf['m_tgibs'])

print(f'Steps matched  : {len(y_true)}  (pos={sum(y_true)}, neg={len(y_true)-sum(y_true)})')
print(f'AUROC GCN-GIBS : {roc_auc_score(y_true, s_gcn):.4f}')
print(f'AUROC TF-GIBS  : {roc_auc_score(y_true, s_tf):.4f}')

TF-GIBS inference: 100%|██████████| 9/9 [00:02<00:00,  3.43task/s]

Runs scored (TF-GIBS): 180
Steps matched  : 2857  (pos=210, neg=2647)
AUROC GCN-GIBS : 0.7615
AUROC TF-GIBS  : 0.6105


## 3. TF-GIBS v2 — Soft Masks + User/Policy Features

Improvements over TF-GIBS v1:
- **Soft MCS mask**: continuous NLI score instead of binary threshold
- **User alignment**: NLI(op_text, last_user_msg)
- **Policy compliance**: NLI(op_text, policy)

GCN cache untouched. New data saved to `tgibs_train_data.pkl`.

In [171]:
# ── TF-v2 Training Data: soft masks + user/policy features ───────────────────
TGIBS_DATA_PATH = DRIVE_BASE + '/tgibs_train_data.pkl'

policy_text = (info.get('environment_info') or {}).get('policy', '')
print(f'Policy text length: {len(policy_text)} chars')

def get_last_user_text(nodes, before_idx):
    for n in reversed(nodes[:before_idx]):
        if n['type'] == 'UserTurn':
            return n.get('op_text') or ''
    return ''

def build_tgibs_data(tid):
    task      = tasks_map[tid]
    all_sims  = sims_by_task.get(tid, [])
    pass_sims = [s for s in all_sims if (s.get('reward_info') or {}).get('reward', 0) >= 0.5]
    if len(pass_sims) < 2: return []
    ref_texts = []
    for ps in pass_sims:
        try:
            sg = build_step_graph(ps, info, task)
            ref_texts += [n['op_text'] for n in sg['nodes']
                          if n['type'] == 'AgentStep' and n.get('op_text')]
        except Exception:
            continue
    if not ref_texts: return []
    results = []
    for run in all_sims:
        try:
            sg   = build_step_graph(run, info, task)
            deps = get_deps(sg)
        except Exception:
            continue
        nodes = sg['nodes']
        agent_nodes = [(i, n) for i, n in enumerate(nodes) if n['type'] == 'AgentStep']
        if not agent_nodes: continue
        q_texts   = [n['op_text'] or '' for _, n in agent_nodes]
        user_msgs = [get_last_user_text(nodes, i) for i, _ in agent_nodes]
        # Prefetch all NLI pairs
        pairs = ([(qt, rt) for qt in q_texts for rt in ref_texts] +
                 [(qt, um) for qt, um in zip(q_texts, user_msgs) if um] +
                 [(qt, policy_text) for qt in q_texts if policy_text])
        sim.prefetch(pairs + [(b, a) for a, b in pairs])
        # Soft mask: continuous score, no threshold
        soft_mask    = [max(sim(qt, rt) for rt in ref_texts) for qt in q_texts]
        user_align   = [sim(qt, um) if um else 0.5 for qt, um in zip(q_texts, user_msgs)]
        policy_score = [sim(qt, policy_text) if policy_text else 0.5 for qt in q_texts]
        # Augmented features: [MiniLM(384) | user_align | policy_score] = 386-dim
        x_base  = encode_texts(q_texts)
        x_extra = torch.tensor([[ua, ps] for ua, ps in zip(user_align, policy_score)])
        x_aug   = torch.cat([x_base, x_extra], dim=1)
        # Adjacency
        id2pos = {n['id']: pos for pos, (_, n) in enumerate(agent_nodes)}
        ep     = [(pos, pos+1) for pos in range(len(agent_nodes)-1)]
        for _, n in agent_nodes:
            ds = deps.get(n['id'])
            if ds and ds in id2pos and n['id'] in id2pos:
                ep.append((id2pos[ds], id2pos[n['id']]))
        adj = build_adj_norm(len(agent_nodes), ep)
        results.append((x_aug, adj, torch.tensor(soft_mask)))
    return results

if os.path.exists(TGIBS_DATA_PATH):
    with open(TGIBS_DATA_PATH, 'rb') as f:
        all_tgibs_data = pickle.load(f)
    print(f'Loaded cached TF-v2 data: {len(all_tgibs_data)} runs')
else:
    all_tgibs_data = []
    for tid in tqdm(TRAIN_TASKS, desc='Building TF-v2 data'):
        all_tgibs_data.extend(build_tgibs_data(tid))
    with open(TGIBS_DATA_PATH, 'wb') as f:
        pickle.dump(all_tgibs_data, f)
    print(f'Built and cached: {len(all_tgibs_data)} runs')

total_nodes = sum(x.shape[0] for x, _, _ in all_tgibs_data)
avg_mask    = sum(m.mean().item() for _, _, m in all_tgibs_data) / len(all_tgibs_data)
print(f'Nodes: {total_nodes}  |  Avg soft mask: {avg_mask:.3f}  |  Feature dim: {all_tgibs_data[0][0].shape[1]}')

Policy text length: 7676 chars
Loaded cached TF-v2 data: 300 runs
Nodes: 5160  |  Avg soft mask: 0.910  |  Feature dim: 386


In [172]:
# ── TF-GIBS v2 Model (386-dim input) ─────────────────────────────────────────
TGIBS_V2_MODEL_PATH = DRIVE_BASE + '/tgibs_v2_model.pt'

class TransformerGIBSv2(nn.Module):
    def __init__(self, embed_dim=386, d_model=128, nhead=4, num_layers=2, dropout=0.2):
        super().__init__()
        self.proj    = nn.Linear(embed_dim, d_model)
        enc_layer    = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dropout=dropout,
            batch_first=True, dim_feedforward=d_model*4
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x):
        n = x.size(0)
        x = self.proj(x)
        causal_mask = torch.triu(torch.ones(n, n, device=x.device), diagonal=1).bool()
        out = self.transformer(x.unsqueeze(0), mask=causal_mask, is_causal=True)
        return torch.sigmoid(self.head(out.squeeze(0))).squeeze(-1)

tgibs_v2_model = TransformerGIBSv2().to(DEVICE)
print(f'TF-GIBSv2 params: {sum(p.numel() for p in tgibs_v2_model.parameters()):,}')

TF-GIBSv2 params: 446,209


In [173]:
# ── TF-GIBS v2 Training ───────────────────────────────────────────────────────
# To retrain: import os; os.remove(TGIBS_V2_MODEL_PATH)

if os.path.exists(TGIBS_V2_MODEL_PATH):
    tgibs_v2_model.load_state_dict(torch.load(TGIBS_V2_MODEL_PATH, map_location=DEVICE))
    tgibs_v2_model.eval()
    print(f'Loaded saved model from {TGIBS_V2_MODEL_PATH}')
else:
    optimizer = Adam(tgibs_v2_model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
    tgibs_v2_model.train()
    for epoch in range(1, 31):
        total_loss = 0.0
        random.shuffle(all_tgibs_data)
        for x, _, mask in all_tgibs_data:
            x, mask = x.to(DEVICE), mask.to(DEVICE)
            optimizer.zero_grad()
            p    = tgibs_v2_model(x)
            loss = ib_loss(p, mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(tgibs_v2_model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if epoch % 5 == 0:
            print(f'Epoch {epoch:3d}/30  loss={total_loss/len(all_tgibs_data):.4f}')
    torch.save(tgibs_v2_model.state_dict(), TGIBS_V2_MODEL_PATH)
    print(f'Training complete. Saved to {TGIBS_V2_MODEL_PATH}')

Loaded saved model from /home/svelag11/agentic/tgibs_v2_model.pt


In [174]:
# ── TF-GIBS v2 Inference + AUROC ─────────────────────────────────────────────
policy_text = (info.get('environment_info') or {}).get('policy', '')
tgibs_v2_model.eval()
all_consensus_tfv2 = {}

with tqdm(total=len(TASKS), desc='TF-GIBSv2 inference', unit='task') as pbar:
    for tid in TASKS:
        task     = tasks_map[tid]
        all_sims = sims_by_task.get(tid, [])
        for run in all_sims:
            try:
                sg = build_step_graph(run, info, task)
            except Exception:
                continue
            nodes = sg['nodes']
            agent_nodes_with_idx = [(i, n) for i, n in enumerate(nodes)
                                    if n['type'] == 'AgentStep']
            if not agent_nodes_with_idx: continue

            q_texts   = [n.get('op_text') or '' for _, n in agent_nodes_with_idx]
            user_msgs = [get_last_user_text(nodes, i) for i, _ in agent_nodes_with_idx]
            user_align   = [sim(qt, um) if um else 0.5 for qt, um in zip(q_texts, user_msgs)]
            policy_score = [sim(qt, policy_text) if policy_text else 0.5 for qt in q_texts]

            x_base  = encode_texts(q_texts)
            x_extra = torch.tensor([[ua, ps] for ua, ps in zip(user_align, policy_score)])
            x_aug   = torch.cat([x_base, x_extra], dim=1).to(DEVICE)

            with torch.no_grad():
                p = tgibs_v2_model(x_aug)

            orig_indices = [i for i, _ in agent_nodes_with_idx]
            all_consensus_tfv2[run['id']] = {
                orig_idx: {'m_tfv2': float(p[pos])}
                for pos, orig_idx in enumerate(orig_indices)
            }
        pbar.update(1)

print(f'Runs scored (TF-GIBSv2): {len(all_consensus_tfv2)}')

y_true, s_tfv2 = [], []
for row in annots:
    sd = all_consensus_tfv2.get(row['run_id'], {}).get(int(row['step_idx']))
    if sd is None: continue
    y_true.append(1 if row['label'] in ('incorrect', 'tool_error') else 0)
    s_tfv2.append(1 - sd['m_tfv2'])

print(f'Steps matched    : {len(y_true)}  (pos={sum(y_true)}, neg={len(y_true)-sum(y_true)})')
print(f'AUROC TF-GIBS v2 : {roc_auc_score(y_true, s_tfv2):.4f}  <- soft masks + user/policy NLI features')
print(f'--- known results ---')
print(f'GCN-GIBS         :  0.7615')
print(f'TF-GIBS v1       :  0.6105')
print(f'NA-MCS           :  0.8101')


TF-GIBSv2 inference: 100%|██████████| 9/9 [00:02<00:00,  3.46task/s]

Runs scored (TF-GIBSv2): 180
Steps matched    : 2857  (pos=210, neg=2647)
AUROC TF-GIBS v2 : 0.6943  <- soft masks + user/policy NLI features
--- known results ---
GCN-GIBS         :  0.7615
TF-GIBS v1       :  0.6105
NA-MCS           :  0.8101


## 4. TF-GIBS v3 — Full Dialogue Sequence

Full conversation modeled as a sequence:
```
[policy] [user_0] [agent_0] [agent_1] [user_1] [agent_2] ...
```
- Policy as position 0 — all tokens attend back to it
- User turns explicitly in sequence — richer than 2 scalar NLI features
- Type embeddings: policy=0, user=1, agent=2
- Confidence scores output only at agent positions

In [175]:
# ── TF-v3 Training Data: full dialogue sequences ─────────────────────────────
TGIBS_V3_DATA_PATH = DRIVE_BASE + '/tgibs_v3_train_data.pkl'

TYPE_POLICY, TYPE_USER, TYPE_AGENT = 0, 1, 2

# Encode policy once
policy_emb = encode_texts([policy_text])[0]  # (384,)
print(f'Policy embedding shape: {policy_emb.shape}')

def build_tgibs_v3_data(tid):
    task      = tasks_map[tid]
    all_sims  = sims_by_task.get(tid, [])
    pass_sims = [s for s in all_sims if (s.get('reward_info') or {}).get('reward', 0) >= 0.5]
    if len(pass_sims) < 2: return []

    ref_texts = []
    for ps in pass_sims:
        try:
            sg = build_step_graph(ps, info, task)
            ref_texts += [n['op_text'] for n in sg['nodes']
                          if n['type'] == 'AgentStep' and n.get('op_text')]
        except Exception:
            continue
    if not ref_texts: return []

    results = []
    for run in all_sims:
        try:
            sg = build_step_graph(run, info, task)
        except Exception:
            continue

        nodes = sg['nodes']
        agent_nodes = [(i, n) for i, n in enumerate(nodes) if n['type'] == 'AgentStep']
        if not agent_nodes: continue

        # Soft mask for agent steps
        q_texts = [n['op_text'] or '' for _, n in agent_nodes]
        pairs   = [(qt, rt) for qt in q_texts for rt in ref_texts]
        sim.prefetch(pairs + [(b, a) for a, b in pairs])
        soft_mask = [max(sim(qt, rt) for rt in ref_texts) for qt in q_texts]

        # Build full sequence: [policy] + all nodes in order
        seq_texts  = [policy_text]
        seq_types  = [TYPE_POLICY]
        agent_positions = []

        for n in nodes:
            if n['type'] == 'UserTurn':
                seq_texts.append(n.get('op_text') or '')
                seq_types.append(TYPE_USER)
            elif n['type'] == 'AgentStep':
                agent_positions.append(len(seq_texts))
                seq_texts.append(n.get('op_text') or '')
                seq_types.append(TYPE_AGENT)

        # Encode full sequence
        seq_embs = torch.cat([
            policy_emb.unsqueeze(0),
            encode_texts(seq_texts[1:])   # skip policy (already encoded)
        ], dim=0)  # (seq_len, 384)

        agent_mask = torch.zeros(len(seq_texts), dtype=torch.bool)
        for pos in agent_positions:
            agent_mask[pos] = True

        results.append((
            seq_embs,
            torch.tensor(seq_types, dtype=torch.long),
            agent_mask,
            torch.tensor(soft_mask)
        ))
    return results

if os.path.exists(TGIBS_V3_DATA_PATH):
    with open(TGIBS_V3_DATA_PATH, 'rb') as f:
        all_tgibs_v3_data = pickle.load(f)
    print(f'Loaded cached TF-v3 data: {len(all_tgibs_v3_data)} runs')
else:
    all_tgibs_v3_data = []
    for tid in tqdm(TRAIN_TASKS, desc='Building TF-v3 data'):
        all_tgibs_v3_data.extend(build_tgibs_v3_data(tid))
    with open(TGIBS_V3_DATA_PATH, 'wb') as f:
        pickle.dump(all_tgibs_v3_data, f)
    print(f'Built and cached: {len(all_tgibs_v3_data)} runs')

avg_seq  = sum(x.shape[0] for x, _, _, _ in all_tgibs_v3_data) / len(all_tgibs_v3_data)
print(f'Runs: {len(all_tgibs_v3_data)}  |  Avg seq length: {avg_seq:.1f}')

Policy embedding shape: torch.Size([384])
Loaded cached TF-v3 data: 300 runs
Runs: 300  |  Avg seq length: 26.5


In [176]:
# ── TF-GIBS v3 Model — full dialogue with type embeddings ────────────────────
TGIBS_V3_MODEL_PATH = DRIVE_BASE + '/tgibs_v3_model.pt'

class TransformerGIBSv3(nn.Module):
    def __init__(self, embed_dim=384, d_model=128, nhead=4, num_layers=2, dropout=0.2):
        super().__init__()
        self.proj     = nn.Linear(embed_dim, d_model)
        self.type_emb = nn.Embedding(3, d_model)  # policy=0, user=1, agent=2
        enc_layer     = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dropout=dropout,
            batch_first=True, dim_feedforward=d_model*4
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x, token_types, agent_mask):
        # x: (seq_len, 384)  token_types: (seq_len,)  agent_mask: (seq_len,) bool
        n = x.size(0)
        h = self.proj(x) + self.type_emb(token_types)   # (n, d_model)
        causal_mask = torch.triu(torch.ones(n, n, device=x.device), diagonal=1).bool()
        out = self.transformer(h.unsqueeze(0), mask=causal_mask, is_causal=True)
        out = out.squeeze(0)                             # (n, d_model)
        agent_out = out[agent_mask]                      # (n_agents, d_model)
        return torch.sigmoid(self.head(agent_out)).squeeze(-1)  # (n_agents,)

tgibs_v3_model = TransformerGIBSv3().to(DEVICE)
print(f'TF-GIBSv3 params: {sum(p.numel() for p in tgibs_v3_model.parameters()):,}')

TF-GIBSv3 params: 446,337


In [177]:
# ── TF-GIBS v3 Training ───────────────────────────────────────────────────────
# To retrain: import os; os.remove(TGIBS_V3_MODEL_PATH)

if os.path.exists(TGIBS_V3_MODEL_PATH):
    tgibs_v3_model.load_state_dict(torch.load(TGIBS_V3_MODEL_PATH, map_location=DEVICE))
    tgibs_v3_model.eval()
    print(f'Loaded saved model from {TGIBS_V3_MODEL_PATH}')
else:
    optimizer = Adam(tgibs_v3_model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
    tgibs_v3_model.train()
    for epoch in range(1, 31):
        total_loss = 0.0
        random.shuffle(all_tgibs_v3_data)
        for x, ttypes, amask, mask in all_tgibs_v3_data:
            x     = x.to(DEVICE)
            ttypes = ttypes.to(DEVICE)
            amask = amask.to(DEVICE)
            mask  = mask.to(DEVICE)
            optimizer.zero_grad()
            p    = tgibs_v3_model(x, ttypes, amask)
            loss = ib_loss(p, mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(tgibs_v3_model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if epoch % 5 == 0:
            print(f'Epoch {epoch:3d}/30  loss={total_loss/len(all_tgibs_v3_data):.4f}')
    torch.save(tgibs_v3_model.state_dict(), TGIBS_V3_MODEL_PATH)
    print(f'Training complete. Saved to {TGIBS_V3_MODEL_PATH}')

Loaded saved model from /home/svelag11/agentic/tgibs_v3_model.pt


In [178]:
# ── TF-GIBS v3 Inference + AUROC ─────────────────────────────────────────────
policy_text = (info.get('environment_info') or {}).get('policy', '')
tgibs_v3_model.eval()
all_consensus_tfv3 = {}

with tqdm(total=len(TASKS), desc='TF-GIBSv3 inference', unit='task') as pbar:
    for tid in TASKS:
        task     = tasks_map[tid]
        all_sims = sims_by_task.get(tid, [])
        for run in all_sims:
            try:
                sg = build_step_graph(run, info, task)
            except Exception:
                continue
            nodes = sg['nodes']
            agent_nodes_with_idx = [(i, n) for i, n in enumerate(nodes)
                                    if n['type'] == 'AgentStep']
            if not agent_nodes_with_idx: continue

            # Build full sequence
            seq_texts = [policy_text]
            seq_types = [TYPE_POLICY]
            agent_positions = []
            for n in nodes:
                if n['type'] == 'UserTurn':
                    seq_texts.append(n.get('op_text') or '')
                    seq_types.append(TYPE_USER)
                elif n['type'] == 'AgentStep':
                    agent_positions.append(len(seq_texts))
                    seq_texts.append(n.get('op_text') or '')
                    seq_types.append(TYPE_AGENT)

            seq_embs = torch.cat([
                policy_emb.unsqueeze(0),
                encode_texts(seq_texts[1:])
            ], dim=0).to(DEVICE)
            ttypes = torch.tensor(seq_types, dtype=torch.long).to(DEVICE)
            amask  = torch.zeros(len(seq_texts), dtype=torch.bool).to(DEVICE)
            for pos in agent_positions:
                amask[pos] = True

            with torch.no_grad():
                p = tgibs_v3_model(seq_embs, ttypes, amask)

            # Map back to original sg node indices
            orig_indices = [i for i, n in enumerate(nodes) if n['type'] == 'AgentStep']
            all_consensus_tfv3[run['id']] = {
                orig_idx: {'m_tfv3': float(p[pos])}
                for pos, orig_idx in enumerate(orig_indices)
            }
        pbar.update(1)

print(f'Runs scored (TF-GIBSv3): {len(all_consensus_tfv3)}')

y_true, s_tfv3 = [], []
for row in annots:
    sd = all_consensus_tfv3.get(row['run_id'], {}).get(int(row['step_idx']))
    if sd is None: continue
    y_true.append(1 if row['label'] in ('incorrect', 'tool_error') else 0)
    s_tfv3.append(1 - sd['m_tfv3'])

print(f'Steps matched    : {len(y_true)}  (pos={sum(y_true)}, neg={len(y_true)-sum(y_true)})')
print(f'AUROC TF-GIBS v3 : {roc_auc_score(y_true, s_tfv3):.4f}  <- full dialogue sequence')
print(f'--- known results ---')
print(f'GCN-GIBS         :  0.7615')
print(f'TF-GIBS v1       :  0.6105')
print(f'TF-GIBS v2       :  0.6943')
print(f'NA-MCS           :  0.8101')

TF-GIBSv3 inference: 100%|██████████| 9/9 [00:03<00:00,  2.72task/s]

Runs scored (TF-GIBSv3): 180
Steps matched    : 2857  (pos=210, neg=2647)
AUROC TF-GIBS v3 : 0.6635  <- full dialogue sequence
--- known results ---
GCN-GIBS         :  0.7615
TF-GIBS v1       :  0.6105
TF-GIBS v2       :  0.6943
NA-MCS           :  0.8101


## 5. R-GCN + Expanded Training Data

Two improvements over GCN-GIBS baseline (0.7615):
1. **R-GCN**: separate weight matrix per edge type (temporal vs data_dependency)
2. **More data**: 40 training tasks (~800 runs) vs 15 tasks (~300 runs)

R-GCN forward: `h = W_self(h) + W_temporal(A_t @ h) + W_datadep(A_d @ h)`

In [179]:
# ── R-GCN Training Data: typed adjacency matrices + expanded tasks ────────────
RGCN_DATA_PATH  = DRIVE_BASE + '/gibs_rgcn_data.pkl'

def build_adj_typed(n, ep_temporal, ep_datadep):
    """Returns (A_temporal_norm, A_datadep_norm) each (n,n)."""
    adjs = []
    for ep in [ep_temporal, ep_datadep]:
        A = torch.zeros(n, n)
        for i, j in ep:
            A[i, j] = 1.0; A[j, i] = 1.0
        A = A + torch.eye(n)
        D = A.sum(1)
        D_inv_sqrt = torch.diag(D.pow(-0.5))
        adjs.append(D_inv_sqrt @ A @ D_inv_sqrt)
    return adjs  # [A_temporal, A_datadep]

def build_rgcn_data(tid):
    task      = tasks_map[tid]
    all_sims  = sims_by_task.get(tid, [])
    pass_sims = [s for s in all_sims if (s.get('reward_info') or {}).get('reward', 0) >= 0.5]
    if len(pass_sims) < 2: return []

    ref_texts = []
    for ps in pass_sims:
        try:
            sg = build_step_graph(ps, info, task)
            ref_texts += [n['op_text'] for n in sg['nodes']
                          if n['type'] == 'AgentStep' and n.get('op_text')]
        except Exception:
            continue
    if not ref_texts: return []

    results = []
    for run in all_sims:
        try:
            sg = build_step_graph(run, info, task)
        except Exception:
            continue

        nodes       = sg['nodes']
        agent_nodes = [(i, n) for i, n in enumerate(nodes) if n['type'] == 'AgentStep']
        if not agent_nodes: continue

        q_texts = [n['op_text'] or '' for _, n in agent_nodes]

        # Prefetch NLI
        pairs = [(qt, rt) for qt in q_texts for rt in ref_texts]
        sim.prefetch(pairs + [(b, a) for a, b in pairs])

        # Consensus mask (hard)
        mask = [1.0 if max(sim(qt, rt) for rt in ref_texts) >= MASK_THRESH else 0.0
                for qt in q_texts]

        x = encode_texts(q_texts)

        # Typed edges
        id2pos   = {n['id']: pos for pos, (_, n) in enumerate(agent_nodes)}
        dep_map  = {e['target']: e['source'] for e in sg['edges'] if e['type'] == 'data_dependency'}

        ep_t = [(pos, pos+1) for pos in range(len(agent_nodes)-1)]
        ep_d = []
        for _, n in agent_nodes:
            src = dep_map.get(n['id'])
            if src and src in id2pos and n['id'] in id2pos:
                ep_d.append((id2pos[src], id2pos[n['id']]))

        adjs = build_adj_typed(len(agent_nodes), ep_t, ep_d)
        results.append((x, adjs, torch.tensor(mask)))
    return results

if os.path.exists(RGCN_DATA_PATH):
    with open(RGCN_DATA_PATH, 'rb') as f:
        all_rgcn_data = pickle.load(f)
    print(f'Loaded cached R-GCN data: {len(all_rgcn_data)} runs')
else:
    all_rgcn_data = []
    for tid in tqdm(TRAIN_TASKS, desc='Building R-GCN data'):
        all_rgcn_data.extend(build_rgcn_data(tid))
    with open(RGCN_DATA_PATH, 'wb') as f:
        pickle.dump(all_rgcn_data, f)
    print(f'Built and cached: {len(all_rgcn_data)} runs')

total_nodes = sum(x.shape[0] for x, _, _ in all_rgcn_data)
pos_ratio   = sum(m.mean().item() for _, _, m in all_rgcn_data) / len(all_rgcn_data)
print(f'Training graphs : {len(all_rgcn_data)}  |  Total nodes: {total_nodes}  |  Avg mask: {pos_ratio:.2f}')


Loaded cached R-GCN data: 800 runs
Training graphs : 800  |  Total nodes: 10896  |  Avg mask: 0.93


In [180]:
# ── R-GCN Model ───────────────────────────────────────────────────────────────
RGCN_MODEL_PATH = DRIVE_BASE + '/rgcn_model.pt'

class RGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim, n_relations=2):
        super().__init__()
        self.W      = nn.ModuleList([nn.Linear(in_dim, out_dim, bias=False)
                                     for _ in range(n_relations)])
        self.W_self = nn.Linear(in_dim, out_dim)

    def forward(self, x, adjs):
        # adjs = [A_temporal, A_datadep]
        out = self.W_self(x)
        for W_r, A_r in zip(self.W, adjs):
            out = out + W_r(A_r @ x)
        return F.relu(out)


class RGCNModel(nn.Module):
    def __init__(self, in_dim=384, hidden=128, n_relations=2):
        super().__init__()
        self.layer1 = RGCNLayer(in_dim, hidden, n_relations)
        self.layer2 = RGCNLayer(hidden, hidden, n_relations)
        self.head   = nn.Linear(hidden, 1)

    def forward(self, x, adjs):
        h = self.layer1(x, adjs)
        h = self.layer2(h, adjs)
        return torch.sigmoid(self.head(h)).squeeze(-1)

rgcn_model = RGCNModel().to(DEVICE)
print(f'R-GCN params: {sum(p.numel() for p in rgcn_model.parameters()):,}')


R-GCN params: 196,993


In [181]:
# ── R-GCN Training ────────────────────────────────────────────────────────────
# To retrain: import os; os.remove(RGCN_MODEL_PATH)

if os.path.exists(RGCN_MODEL_PATH):
    rgcn_model.load_state_dict(torch.load(RGCN_MODEL_PATH, map_location=DEVICE))
    rgcn_model.eval()
    print(f'Loaded saved model from {RGCN_MODEL_PATH}')
else:
    optimizer = Adam(rgcn_model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
    rgcn_model.train()
    for epoch in range(1, 51):
        total_loss = 0.0
        random.shuffle(all_rgcn_data)
        for x, adjs, mask in all_rgcn_data:
            x    = x.to(DEVICE)
            adjs = [a.to(DEVICE) for a in adjs]
            mask = mask.to(DEVICE)
            optimizer.zero_grad()
            p    = rgcn_model(x, adjs)
            loss = ib_loss(p, mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(rgcn_model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if epoch % 10 == 0:
            print(f'Epoch {epoch:3d}/50  loss={total_loss/len(all_rgcn_data):.4f}')
    torch.save(rgcn_model.state_dict(), RGCN_MODEL_PATH)
    print(f'Training complete. Saved to {RGCN_MODEL_PATH}')


Loaded saved model from /home/svelag11/agentic/rgcn_model.pt


In [182]:
# ── R-GCN Inference + AUROC ───────────────────────────────────────────────────
rgcn_model.eval()
all_consensus_rgcn = {}

with tqdm(total=len(TASKS), desc='R-GCN inference', unit='task') as pbar:
    for tid in TASKS:
        task     = tasks_map[tid]
        all_sims = sims_by_task.get(tid, [])
        for run in all_sims:
            try:
                sg = build_step_graph(run, info, task)
            except Exception:
                continue
            nodes = sg['nodes']
            agent_nodes_with_idx = [(i, n) for i, n in enumerate(nodes)
                                    if n['type'] == 'AgentStep']
            if not agent_nodes_with_idx: continue

            q_texts = [n.get('op_text') or '' for _, n in agent_nodes_with_idx]
            x       = encode_texts(q_texts).to(DEVICE)

            id2pos  = {n['id']: pos for pos, (_, n) in enumerate(agent_nodes_with_idx)}
            dep_map = {e['target']: e['source'] for e in sg['edges'] if e['type'] == 'data_dependency'}
            n_steps = len(agent_nodes_with_idx)

            ep_t = [(pos, pos+1) for pos in range(n_steps-1)]
            ep_d = []
            for _, n in agent_nodes_with_idx:
                src = dep_map.get(n['id'])
                if src and src in id2pos and n['id'] in id2pos:
                    ep_d.append((id2pos[src], id2pos[n['id']]))

            adjs = [a.to(DEVICE) for a in build_adj_typed(n_steps, ep_t, ep_d)]

            with torch.no_grad():
                p = rgcn_model(x, adjs)

            orig_indices = [i for i, _ in agent_nodes_with_idx]
            all_consensus_rgcn[run['id']] = {
                orig_idx: {'m_rgcn': float(p[pos])}
                for pos, orig_idx in enumerate(orig_indices)
            }
        pbar.update(1)

print(f'Runs scored (R-GCN): {len(all_consensus_rgcn)}')

y_true, s_rgcn = [], []
for row in annots:
    sd = all_consensus_rgcn.get(row['run_id'], {}).get(int(row['step_idx']))
    if sd is None: continue
    y_true.append(1 if row['label'] in ('incorrect', 'tool_error') else 0)
    s_rgcn.append(1 - sd['m_rgcn'])

print(f'Steps matched : {len(y_true)}  (pos={sum(y_true)}, neg={len(y_true)-sum(y_true)})')
print()
print(f'AUROC R-GCN           : {roc_auc_score(y_true, s_rgcn):.4f}  <- typed edges + 40 tasks')
print(f'--- known results ---')
print(f'NA-MCS                :  0.8101')
print(f'GCN-GIBS              :  0.7615  (15 tasks, merged edges)')
print(f'TF-GIBS v2            :  0.6943')
print(f'TF-GIBS v3            :  0.6635')
print(f'TF-GIBS v1            :  0.6105')


R-GCN inference: 100%|██████████| 9/9 [00:02<00:00,  3.54task/s]

Runs scored (R-GCN): 180
Steps matched : 2857  (pos=210, neg=2647)

AUROC R-GCN           : 0.7231  <- typed edges + 40 tasks
--- known results ---
NA-MCS                :  0.8101
GCN-GIBS              :  0.7615  (15 tasks, merged edges)
TF-GIBS v2            :  0.6943
TF-GIBS v3            :  0.6635
TF-GIBS v1            :  0.6105


In [183]:
# ── TF-v2 Expanded: build training data from 40 tasks ────────────────────────
TGIBS_EXP_DATA_PATH = DRIVE_BASE + '/tgibs_v2_expanded_data.pkl'

if os.path.exists(TGIBS_EXP_DATA_PATH):
    with open(TGIBS_EXP_DATA_PATH, 'rb') as f:
        all_tgibs_exp_data = pickle.load(f)
    print(f'Loaded cached TF-v2 expanded data: {len(all_tgibs_exp_data)} runs')
else:
    all_tgibs_exp_data = []
    for tid in tqdm(TRAIN_TASKS_EXPANDED, desc='Building TF-v2 expanded data'):
        all_tgibs_exp_data.extend(build_tgibs_data(tid))
    with open(TGIBS_EXP_DATA_PATH, 'wb') as f:
        pickle.dump(all_tgibs_exp_data, f)
    print(f'Built and cached: {len(all_tgibs_exp_data)} runs')

total_nodes = sum(x.shape[0] for x, _, _ in all_tgibs_exp_data)
avg_mask    = sum(m.mean().item() for _, _, m in all_tgibs_exp_data) / len(all_tgibs_exp_data)
print(f'Nodes: {total_nodes}  |  Avg soft mask: {avg_mask:.3f}')


Loaded cached TF-v2 expanded data: 800 runs
Nodes: 10896  |  Avg soft mask: 0.942


In [184]:
# ── TF-v2 Expanded: train ─────────────────────────────────────────────────────
TGIBS_V2_EXP_MODEL_PATH = DRIVE_BASE + '/tgibs_v2_expanded_model.pt'
# AdamW + warmup/cosine + lam=2.0 + 100 epochs
# To retrain: import os; os.remove(TGIBS_V2_EXP_MODEL_PATH)

tgibs_v2_exp_model = TransformerGIBSv2().to(DEVICE)

if os.path.exists(TGIBS_V2_EXP_MODEL_PATH):
    tgibs_v2_exp_model.load_state_dict(torch.load(TGIBS_V2_EXP_MODEL_PATH, map_location=DEVICE))
    tgibs_v2_exp_model.eval()
    print(f'Loaded saved model from {TGIBS_V2_EXP_MODEL_PATH}')
else:
    TOTAL_EPOCHS, WARMUP_EPOCHS = 100, 10
    optimizer    = torch.optim.AdamW(tgibs_v2_exp_model.parameters(), lr=3e-4, weight_decay=0.01)
    warmup_sched = torch.optim.lr_scheduler.LambdaLR(optimizer,
                       lambda e: (e + 1) / WARMUP_EPOCHS if e < WARMUP_EPOCHS else 1.0)
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS - WARMUP_EPOCHS)
    scheduler    = torch.optim.lr_scheduler.SequentialLR(
                       optimizer, [warmup_sched, cosine_sched], milestones=[WARMUP_EPOCHS])
    tgibs_v2_exp_model.train()
    for epoch in range(1, TOTAL_EPOCHS + 1):
        total_loss = 0.0
        random.shuffle(all_tgibs_exp_data)
        for x, _, mask in all_tgibs_exp_data:
            x, mask = x.to(DEVICE), mask.to(DEVICE)
            optimizer.zero_grad()
            p    = tgibs_v2_exp_model(x)
            loss = ib_loss(p, mask, lam=2.0)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(tgibs_v2_exp_model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if epoch % 10 == 0:
            lr = scheduler.get_last_lr()[0]
            print(f'Epoch {epoch:3d}/{TOTAL_EPOCHS}  loss={total_loss/len(all_tgibs_exp_data):.4f}  lr={lr:.2e}')
    torch.save(tgibs_v2_exp_model.state_dict(), TGIBS_V2_EXP_MODEL_PATH)
    print(f'Training complete. Saved to {TGIBS_V2_EXP_MODEL_PATH}')


Epoch  10/100  loss=0.9158  lr=3.00e-04
Epoch  20/100  loss=0.7324  lr=2.91e-04
Epoch  30/100  loss=0.6410  lr=2.65e-04
Epoch  40/100  loss=0.6072  lr=2.25e-04
Epoch  50/100  loss=0.5803  lr=1.76e-04
Epoch  60/100  loss=0.5723  lr=1.24e-04
Epoch  70/100  loss=0.5641  lr=7.50e-05
Epoch  80/100  loss=0.5590  lr=3.51e-05
Epoch  90/100  loss=0.5575  lr=9.05e-06
Epoch 100/100  loss=0.5560  lr=0.00e+00
Training complete. Saved to /home/svelag11/agentic/tgibs_v2_expanded_model.pt


In [185]:
# ── TF-v2 Expanded: Inference + AUROC ────────────────────────────────────────
policy_text = (info.get('environment_info') or {}).get('policy', '')
tgibs_v2_exp_model.eval()
all_consensus_tfv2_exp = {}

with tqdm(total=len(TASKS), desc='TF-v2-exp inference', unit='task') as pbar:
    for tid in TASKS:
        task     = tasks_map[tid]
        all_sims = sims_by_task.get(tid, [])
        for run in all_sims:
            try:
                sg = build_step_graph(run, info, task)
            except Exception:
                continue
            nodes = sg['nodes']
            agent_nodes_with_idx = [(i, n) for i, n in enumerate(nodes)
                                    if n['type'] == 'AgentStep']
            if not agent_nodes_with_idx: continue

            q_texts   = [n.get('op_text') or '' for _, n in agent_nodes_with_idx]
            user_msgs = [get_last_user_text(nodes, i) for i, _ in agent_nodes_with_idx]
            user_align   = [sim(qt, um) if um else 0.5 for qt, um in zip(q_texts, user_msgs)]
            policy_score = [sim(qt, policy_text) if policy_text else 0.5 for qt in q_texts]

            x_base  = encode_texts(q_texts)
            x_extra = torch.tensor([[ua, ps] for ua, ps in zip(user_align, policy_score)])
            x_aug   = torch.cat([x_base, x_extra], dim=1).to(DEVICE)

            with torch.no_grad():
                p = tgibs_v2_exp_model(x_aug)

            orig_indices = [i for i, _ in agent_nodes_with_idx]
            all_consensus_tfv2_exp[run['id']] = {
                orig_idx: {'m': float(p[pos])}
                for pos, orig_idx in enumerate(orig_indices)
            }
        pbar.update(1)

print(f'Runs scored: {len(all_consensus_tfv2_exp)}')

y_true, s_tfv2_exp = [], []
for row in annots:
    sd = all_consensus_tfv2_exp.get(row['run_id'], {}).get(int(row['step_idx']))
    if sd is None: continue
    y_true.append(1 if row['label'] in ('incorrect', 'tool_error') else 0)
    s_tfv2_exp.append(1 - sd['m'])

print(f'Steps matched       : {len(y_true)}  (pos={sum(y_true)}, neg={len(y_true)-sum(y_true)})')
print()
print(f'AUROC TF-v2 expanded: {roc_auc_score(y_true, s_tfv2_exp):.4f}  <- soft masks + NLI + 40 tasks')
print(f'--- known results ---')
print(f'NA-MCS              :  0.8101')
print(f'GCN-GIBS            :  0.7615  (15 tasks)')
print(f'TF-GIBS v2          :  0.6943  (15 tasks)')


TF-v2-exp inference: 100%|██████████| 9/9 [00:02<00:00,  3.40task/s]

Runs scored: 180
Steps matched       : 2857  (pos=210, neg=2647)

AUROC TF-v2 expanded: 0.7002  <- soft masks + NLI + 40 tasks
--- known results ---
NA-MCS              :  0.8101
GCN-GIBS            :  0.7615  (15 tasks)
TF-GIBS v2          :  0.6943  (15 tasks)


In [186]:
# ── TF-v3 Expanded: build training data from 40 tasks ────────────────────────
TGIBS_V3_EXP_DATA_PATH = DRIVE_BASE + '/tgibs_v3_expanded_data.pkl'

if os.path.exists(TGIBS_V3_EXP_DATA_PATH):
    with open(TGIBS_V3_EXP_DATA_PATH, 'rb') as f:
        all_tgibs_v3_exp_data = pickle.load(f)
    print(f'Loaded cached TF-v3 expanded data: {len(all_tgibs_v3_exp_data)} runs')
else:
    all_tgibs_v3_exp_data = []
    for tid in tqdm(TRAIN_TASKS_EXPANDED, desc='Building TF-v3 expanded data'):
        all_tgibs_v3_exp_data.extend(build_tgibs_v3_data(tid))
    with open(TGIBS_V3_EXP_DATA_PATH, 'wb') as f:
        pickle.dump(all_tgibs_v3_exp_data, f)
    print(f'Built and cached: {len(all_tgibs_v3_exp_data)} runs')

total_nodes = sum(seq.shape[0] for seq, _, _, _ in all_tgibs_v3_exp_data)
avg_mask    = sum(m.float().mean().item() for _, _, _, m in all_tgibs_v3_exp_data) / len(all_tgibs_v3_exp_data)
print(f'Sequences: {len(all_tgibs_v3_exp_data)}  |  Total tokens: {total_nodes}  |  Avg mask: {avg_mask:.3f}')


Building TF-v3 expanded data: 100%|██████████| 40/40 [08:45<00:00, 13.14s/it]


Built and cached: 800 runs
Sequences: 800  |  Total tokens: 17384  |  Avg mask: 0.942


In [187]:
# ── TF-v3 Expanded: train ─────────────────────────────────────────────────────
TGIBS_V3_EXP_MODEL_PATH = DRIVE_BASE + '/tgibs_v3_expanded_model.pt'
# AdamW + warmup/cosine + lam=2.0 + 100 epochs
# To retrain: import os; os.remove(TGIBS_V3_EXP_MODEL_PATH)

tgibs_v3_exp_model = TransformerGIBSv3().to(DEVICE)

if os.path.exists(TGIBS_V3_EXP_MODEL_PATH):
    tgibs_v3_exp_model.load_state_dict(torch.load(TGIBS_V3_EXP_MODEL_PATH, map_location=DEVICE))
    tgibs_v3_exp_model.eval()
    print(f'Loaded saved model from {TGIBS_V3_EXP_MODEL_PATH}')
else:
    TOTAL_EPOCHS, WARMUP_EPOCHS = 100, 10
    optimizer    = torch.optim.AdamW(tgibs_v3_exp_model.parameters(), lr=3e-4, weight_decay=0.01)
    warmup_sched = torch.optim.lr_scheduler.LambdaLR(optimizer,
                       lambda e: (e + 1) / WARMUP_EPOCHS if e < WARMUP_EPOCHS else 1.0)
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS - WARMUP_EPOCHS)
    scheduler    = torch.optim.lr_scheduler.SequentialLR(
                       optimizer, [warmup_sched, cosine_sched], milestones=[WARMUP_EPOCHS])
    tgibs_v3_exp_model.train()
    for epoch in range(1, TOTAL_EPOCHS + 1):
        total_loss = 0.0
        random.shuffle(all_tgibs_v3_exp_data)
        for seq_embs, token_types, agent_mask, soft_mask in all_tgibs_v3_exp_data:
            seq_embs    = seq_embs.to(DEVICE)
            token_types = token_types.to(DEVICE)
            agent_mask  = agent_mask.to(DEVICE)
            soft_mask   = soft_mask.to(DEVICE)
            optimizer.zero_grad()
            p    = tgibs_v3_exp_model(seq_embs, token_types, agent_mask)
            loss = ib_loss(p, soft_mask, lam=2.0)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(tgibs_v3_exp_model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if epoch % 10 == 0:
            lr = scheduler.get_last_lr()[0]
            print(f'Epoch {epoch:3d}/{TOTAL_EPOCHS}  loss={total_loss/len(all_tgibs_v3_exp_data):.4f}  lr={lr:.2e}')
    torch.save(tgibs_v3_exp_model.state_dict(), TGIBS_V3_EXP_MODEL_PATH)
    print(f'Training complete. Saved to {TGIBS_V3_EXP_MODEL_PATH}')


Epoch  10/100  loss=1.0219  lr=3.00e-04
Epoch  20/100  loss=0.8470  lr=2.91e-04
Epoch  30/100  loss=0.7886  lr=2.65e-04
Epoch  40/100  loss=0.7244  lr=2.25e-04
Epoch  50/100  loss=0.6846  lr=1.76e-04
Epoch  60/100  loss=0.6645  lr=1.24e-04
Epoch  70/100  loss=0.6487  lr=7.50e-05
Epoch  80/100  loss=0.6355  lr=3.51e-05
Epoch  90/100  loss=0.6280  lr=9.05e-06
Epoch 100/100  loss=0.6245  lr=0.00e+00
Training complete. Saved to /home/svelag11/agentic/tgibs_v3_expanded_model.pt


In [188]:
# ── TF-v3 Expanded: Inference + AUROC ────────────────────────────────────────
policy_text = (info.get('environment_info') or {}).get('policy', '')
tgibs_v3_exp_model.eval()
all_consensus_tfv3_exp = {}

with tqdm(total=len(TASKS), desc='TF-v3-exp inference', unit='task') as pbar:
    for tid in TASKS:
        task     = tasks_map[tid]
        all_sims = sims_by_task.get(tid, [])
        for run in all_sims:
            try:
                sg = build_step_graph(run, info, task)
            except Exception:
                continue
            nodes = sg['nodes']
            agent_nodes_with_idx = [(i, n) for i, n in enumerate(nodes)
                                    if n['type'] == 'AgentStep']
            if not agent_nodes_with_idx: continue

            seq_texts = [policy_text]
            seq_types = [TYPE_POLICY]
            agent_positions = []
            for n in nodes:
                if n['type'] == 'UserTurn':
                    seq_texts.append(n.get('op_text') or '')
                    seq_types.append(TYPE_USER)
                elif n['type'] == 'AgentStep':
                    agent_positions.append(len(seq_texts))
                    seq_texts.append(n.get('op_text') or '')
                    seq_types.append(TYPE_AGENT)

            seq_embs = torch.cat([
                policy_emb.unsqueeze(0),
                encode_texts(seq_texts[1:])
            ], dim=0).to(DEVICE)
            ttypes = torch.tensor(seq_types, dtype=torch.long).to(DEVICE)
            amask  = torch.zeros(len(seq_texts), dtype=torch.bool).to(DEVICE)
            for pos in agent_positions:
                amask[pos] = True

            with torch.no_grad():
                p = tgibs_v3_exp_model(seq_embs, ttypes, amask)

            orig_indices = [i for i, n in enumerate(nodes) if n['type'] == 'AgentStep']
            all_consensus_tfv3_exp[run['id']] = {
                orig_idx: {'m': float(p[pos])}
                for pos, orig_idx in enumerate(orig_indices)
            }
        pbar.update(1)

print(f'Runs scored: {len(all_consensus_tfv3_exp)}')

y_true, s_tfv3_exp = [], []
for row in annots:
    sd = all_consensus_tfv3_exp.get(row['run_id'], {}).get(int(row['step_idx']))
    if sd is None: continue
    y_true.append(1 if row['label'] in ('incorrect', 'tool_error') else 0)
    s_tfv3_exp.append(1 - sd['m'])

print(f'Steps matched       : {len(y_true)}  (pos={sum(y_true)}, neg={len(y_true)-sum(y_true)})')
print()
print(f'AUROC TF-v3 expanded: {roc_auc_score(y_true, s_tfv3_exp):.4f}  <- full dialogue + 40 tasks')
print(f'--- known results ---')
print(f'NA-MCS              :  0.8101')
print(f'GCN-GIBS            :  0.7615  (15 tasks)')
print(f'TF-GIBS v2          :  0.6943  (15 tasks)')
print(f'TF-GIBS v3          :  0.6635  (15 tasks)')


TF-v3-exp inference: 100%|██████████| 9/9 [00:03<00:00,  2.67task/s]

Runs scored: 180
Steps matched       : 2857  (pos=210, neg=2647)

AUROC TF-v3 expanded: 0.6942  <- full dialogue + 40 tasks
--- known results ---
NA-MCS              :  0.8101
GCN-GIBS            :  0.7615  (15 tasks)
TF-GIBS v2          :  0.6943  (15 tasks)
TF-GIBS v3          :  0.6635  (15 tasks)


## 1c. GCN + NLI + Expanded Tasks

Same GCN+NLI (386-dim) but trained on 40 tasks (~800 runs).
Reuses `all_tgibs_exp_data` from TF-v2 expanded cell.

In [189]:
# ── GCN + NLI + Expanded: model ───────────────────────────────────────────────
GCN_NLI_EXP_MODEL_PATH = DRIVE_BASE + '/gibs_gcn_nli_exp_model.pt'

gcn_nli_exp_model = GIBSModel(embed_dim=386).to(DEVICE)
print(f'GCN+NLI-exp params : {sum(p.numel() for p in gcn_nli_exp_model.parameters()):,}')
print(f'Training data      : {len(all_tgibs_exp_data)} runs (40 tasks, 386-dim features)')


GCN+NLI-exp params : 28,865
Training data      : 800 runs (40 tasks, 386-dim features)


In [190]:
# ── GCN + NLI + Expanded: train ───────────────────────────────────────────────
# To retrain: import os; os.remove(GCN_NLI_EXP_MODEL_PATH)

if os.path.exists(GCN_NLI_EXP_MODEL_PATH):
    gcn_nli_exp_model.load_state_dict(torch.load(GCN_NLI_EXP_MODEL_PATH, map_location=DEVICE))
    gcn_nli_exp_model.eval()
    print(f'Loaded saved model from {GCN_NLI_EXP_MODEL_PATH}')
else:
    optimizer = torch.optim.AdamW(gcn_nli_exp_model.parameters(), lr=3e-4, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
    gcn_nli_exp_model.train()
    for epoch in range(1, 51):
        total_loss = 0.0
        random.shuffle(all_tgibs_exp_data)
        for x, adj, mask in all_tgibs_exp_data:
            x, adj, mask = x.to(DEVICE), adj.to(DEVICE), mask.to(DEVICE)
            optimizer.zero_grad()
            p    = gcn_nli_exp_model(x, adj)
            loss = ib_loss(p, mask, lam=2.0)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(gcn_nli_exp_model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if epoch % 10 == 0:
            print(f'Epoch {epoch:3d}/50  loss={total_loss/len(all_tgibs_exp_data):.4f}')
    torch.save(gcn_nli_exp_model.state_dict(), GCN_NLI_EXP_MODEL_PATH)
    print(f'Training complete. Saved to {GCN_NLI_EXP_MODEL_PATH}')


Epoch  10/50  loss=1.0600
Epoch  20/50  loss=0.9482
Epoch  30/50  loss=0.8928
Epoch  40/50  loss=0.8522
Epoch  50/50  loss=0.8448
Training complete. Saved to /home/svelag11/agentic/gibs_gcn_nli_exp_model.pt


In [191]:
# ── GCN + NLI + Expanded: Inference + AUROC ───────────────────────────────────
policy_text = (info.get('environment_info') or {}).get('policy', '')
gcn_nli_exp_model.eval()
all_consensus_gcn_nli_exp = {}

with tqdm(total=len(TASKS), desc='GCN+NLI-exp inference', unit='task') as pbar:
    for tid in TASKS:
        task     = tasks_map[tid]
        all_sims = sims_by_task.get(tid, [])
        for run in all_sims:
            try:
                sg = build_step_graph(run, info, task)
            except Exception:
                continue
            nodes = sg['nodes']
            agent_nodes_with_idx = [(i, n) for i, n in enumerate(nodes)
                                    if n['type'] == 'AgentStep']
            if not agent_nodes_with_idx: continue

            q_texts   = [n.get('op_text') or '' for _, n in agent_nodes_with_idx]
            user_msgs = [get_last_user_text(nodes, i) for i, _ in agent_nodes_with_idx]
            user_align   = [sim(qt, um) if um else 0.5 for qt, um in zip(q_texts, user_msgs)]
            policy_score = [sim(qt, policy_text) if policy_text else 0.5 for qt in q_texts]

            x_base  = encode_texts(q_texts)
            x_extra = torch.tensor([[ua, ps] for ua, ps in zip(user_align, policy_score)])
            x_aug   = torch.cat([x_base, x_extra], dim=1).to(DEVICE)

            id2pos  = {n['id']: pos for pos, (_, n) in enumerate(agent_nodes_with_idx)}
            dep_map = {e['target']: e['source'] for e in sg['edges'] if e['type'] == 'data_dependency'}
            n_steps = len(agent_nodes_with_idx)
            ep = [(pos, pos+1) for pos in range(n_steps-1)]
            for _, n in agent_nodes_with_idx:
                src = dep_map.get(n['id'])
                if src and src in id2pos and n['id'] in id2pos:
                    ep.append((id2pos[src], id2pos[n['id']]))
            adj = build_adj_norm(n_steps, ep).to(DEVICE)

            with torch.no_grad():
                p = gcn_nli_exp_model(x_aug, adj)

            orig_indices = [i for i, _ in agent_nodes_with_idx]
            all_consensus_gcn_nli_exp[run['id']] = {
                orig_idx: {'m': float(p[pos])}
                for pos, orig_idx in enumerate(orig_indices)
            }
        pbar.update(1)

print(f'Runs scored: {len(all_consensus_gcn_nli_exp)}')

y_true, s = [], []
for row in annots:
    sd = all_consensus_gcn_nli_exp.get(row['run_id'], {}).get(int(row['step_idx']))
    if sd is None: continue
    y_true.append(1 if row['label'] in ('incorrect', 'tool_error') else 0)
    s.append(1 - sd['m'])

print(f'Steps matched       : {len(y_true)}  (pos={sum(y_true)}, neg={len(y_true)-sum(y_true)})')
print()
print(f'AUROC GCN+NLI+exp   : {roc_auc_score(y_true, s):.4f}  <- 386-dim + graph + 40 tasks')
print(f'--- known results ---')
print(f'NA-MCS              :  0.8101')
print(f'GCN-GIBS            :  0.7615  (384-dim, 15 tasks)')
print(f'TF-GIBS v2          :  0.6943  (386-dim, no graph, 15 tasks)')


GCN+NLI-exp inference: 100%|██████████| 9/9 [00:02<00:00,  3.51task/s]

Runs scored: 180
Steps matched       : 2857  (pos=210, neg=2647)

AUROC GCN+NLI+exp   : 0.8201  <- 386-dim + graph + 40 tasks
--- known results ---
NA-MCS              :  0.8101
GCN-GIBS            :  0.7615  (384-dim, 15 tasks)
TF-GIBS v2          :  0.6943  (386-dim, no graph, 15 tasks)


## TF-GIBS v4 — Full Dialogue + NLI Features

Combines TF-v3 (full sequence) + TF-v2 (NLI scalars).

All tokens padded to 386-dim:
- policy/user tokens: `[MiniLM(384) | 0 | 0]`
- agent tokens:       `[MiniLM(384) | NLI(op, user_msg) | NLI(op, policy)]`

Single `Linear(386→128)` projection + type_emb. Model identical to v3 otherwise.

In [192]:
# ── TF-v4 Data: full dialogue + NLI scalars on agent tokens ──────────────────
TGIBS_V4_DATA_PATH = DRIVE_BASE + '/tgibs_v4_train_data.pkl'
ZERO_PAD = torch.zeros(2)

def build_tgibs_v4_data(tid):
    task      = tasks_map[tid]
    all_sims  = sims_by_task.get(tid, [])
    pass_sims = [s for s in all_sims if (s.get('reward_info') or {}).get('reward', 0) >= 0.5]
    if len(pass_sims) < 2: return []

    ref_texts = []
    for ps in pass_sims:
        try:
            sg = build_step_graph(ps, info, task)
            ref_texts += [n['op_text'] for n in sg['nodes']
                          if n['type'] == 'AgentStep' and n.get('op_text')]
        except Exception:
            continue
    if not ref_texts: return []

    results = []
    for run in all_sims:
        try:
            sg = build_step_graph(run, info, task)
        except Exception:
            continue
        nodes = sg['nodes']
        agent_nodes = [(i, n) for i, n in enumerate(nodes) if n['type'] == 'AgentStep']
        if not agent_nodes: continue

        q_texts   = [n['op_text'] or '' for _, n in agent_nodes]
        user_msgs = [get_last_user_text(nodes, i) for i, _ in agent_nodes]

        # Prefetch NLI
        pairs = ([(qt, rt) for qt in q_texts for rt in ref_texts] +
                 [(qt, um) for qt, um in zip(q_texts, user_msgs) if um] +
                 [(qt, policy_text) for qt in q_texts if policy_text])
        sim.prefetch(pairs + [(b, a) for a, b in pairs])

        soft_mask    = [max(sim(qt, rt) for rt in ref_texts) for qt in q_texts]
        user_align   = [sim(qt, um) if um else 0.5 for qt, um in zip(q_texts, user_msgs)]
        policy_score = [sim(qt, policy_text) if policy_text else 0.5 for qt in q_texts]

        # Build 386-dim sequence
        seq_texts, seq_types, agent_positions = [policy_text], [TYPE_POLICY], []
        for n in nodes:
            if n['type'] == 'UserTurn':
                seq_texts.append(n.get('op_text') or '')
                seq_types.append(TYPE_USER)
            elif n['type'] == 'AgentStep':
                agent_positions.append(len(seq_texts))
                seq_texts.append(n.get('op_text') or '')
                seq_types.append(TYPE_AGENT)

        base_embs = torch.cat([policy_emb.unsqueeze(0), encode_texts(seq_texts[1:])], dim=0)  # (n, 384)

        # Append NLI scalars: zeros for non-agent, [ua, ps] for agent
        agent_iter = iter(zip(user_align, policy_score))
        rows = []
        for pos, ttype in enumerate(seq_types):
            if ttype == TYPE_AGENT:
                ua, ps = next(agent_iter)
                rows.append(torch.cat([base_embs[pos], torch.tensor([ua, ps])]))  # 386
            else:
                rows.append(torch.cat([base_embs[pos], ZERO_PAD]))  # 386
        seq_embs_v4 = torch.stack(rows)  # (n, 386)

        ttypes   = torch.tensor(seq_types, dtype=torch.long)
        amask    = torch.zeros(len(seq_types), dtype=torch.bool)
        for pos in agent_positions:
            amask[pos] = True
        smask = torch.tensor(soft_mask)

        results.append((seq_embs_v4, ttypes, amask, smask))
    return results

if os.path.exists(TGIBS_V4_DATA_PATH):
    with open(TGIBS_V4_DATA_PATH, 'rb') as f:
        all_tgibs_v4_data = pickle.load(f)
    print(f'Loaded cached TF-v4 data: {len(all_tgibs_v4_data)} runs')
else:
    all_tgibs_v4_data = []
    for tid in tqdm(TRAIN_TASKS, desc='Building TF-v4 data'):
        all_tgibs_v4_data.extend(build_tgibs_v4_data(tid))
    with open(TGIBS_V4_DATA_PATH, 'wb') as f:
        pickle.dump(all_tgibs_v4_data, f)
    print(f'Built and cached: {len(all_tgibs_v4_data)} runs')

print(f'Sequences: {len(all_tgibs_v4_data)}  |  Feature dim: 386')


Building TF-v4 data: 100%|██████████| 15/15 [00:40<00:00,  2.68s/it]

Built and cached: 300 runs
Sequences: 300  |  Feature dim: 386


In [193]:
# ── TF-GIBS v4 Model (386-dim, full dialogue + NLI) ──────────────────────────
TGIBS_V4_MODEL_PATH = DRIVE_BASE + '/tgibs_v4_model.pt'

# Identical to TransformerGIBSv3 but embed_dim=386
class TransformerGIBSv4(nn.Module):
    def __init__(self, embed_dim=386, d_model=128, nhead=4, num_layers=2, dropout=0.2):
        super().__init__()
        self.proj     = nn.Linear(embed_dim, d_model)
        self.type_emb = nn.Embedding(3, d_model)
        enc_layer     = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dropout=dropout,
            batch_first=True, dim_feedforward=d_model*4)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x, token_types, agent_mask):
        n = x.size(0)
        h = self.proj(x) + self.type_emb(token_types)
        causal_mask = torch.triu(torch.ones(n, n, device=x.device), diagonal=1).bool()
        out = self.transformer(h.unsqueeze(0), mask=causal_mask, is_causal=True).squeeze(0)
        return torch.sigmoid(self.head(out[agent_mask])).squeeze(-1)

tgibs_v4_model = TransformerGIBSv4().to(DEVICE)
print(f'TF-GIBSv4 params: {sum(p.numel() for p in tgibs_v4_model.parameters()):,}')


TF-GIBSv4 params: 446,593


In [194]:
# ── TF-GIBS v4 Training (AdamW + warmup + lam=2.0, 100 epochs) ───────────────
# To retrain: import os; os.remove(TGIBS_V4_MODEL_PATH)

if os.path.exists(TGIBS_V4_MODEL_PATH):
    tgibs_v4_model.load_state_dict(torch.load(TGIBS_V4_MODEL_PATH, map_location=DEVICE))
    tgibs_v4_model.eval()
    print(f'Loaded saved model from {TGIBS_V4_MODEL_PATH}')
else:
    TOTAL_EPOCHS, WARMUP_EPOCHS = 100, 10
    optimizer    = torch.optim.AdamW(tgibs_v4_model.parameters(), lr=3e-4, weight_decay=0.01)
    warmup_sched = torch.optim.lr_scheduler.LambdaLR(optimizer,
                       lambda e: (e + 1) / WARMUP_EPOCHS if e < WARMUP_EPOCHS else 1.0)
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS - WARMUP_EPOCHS)
    scheduler    = torch.optim.lr_scheduler.SequentialLR(
                       optimizer, [warmup_sched, cosine_sched], milestones=[WARMUP_EPOCHS])
    tgibs_v4_model.train()
    for epoch in range(1, TOTAL_EPOCHS + 1):
        total_loss = 0.0
        random.shuffle(all_tgibs_v4_data)
        for seq_embs, token_types, agent_mask, soft_mask in all_tgibs_v4_data:
            seq_embs    = seq_embs.to(DEVICE)
            token_types = token_types.to(DEVICE)
            agent_mask  = agent_mask.to(DEVICE)
            soft_mask   = soft_mask.to(DEVICE)
            optimizer.zero_grad()
            p    = tgibs_v4_model(seq_embs, token_types, agent_mask)
            loss = ib_loss(p, soft_mask, lam=2.0)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(tgibs_v4_model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if epoch % 10 == 0:
            lr = scheduler.get_last_lr()[0]
            print(f'Epoch {epoch:3d}/{TOTAL_EPOCHS}  loss={total_loss/len(all_tgibs_v4_data):.4f}  lr={lr:.2e}')
    torch.save(tgibs_v4_model.state_dict(), TGIBS_V4_MODEL_PATH)
    print(f'Training complete. Saved to {TGIBS_V4_MODEL_PATH}')


Epoch  10/100  loss=1.3434  lr=3.00e-04
Epoch  20/100  loss=1.1671  lr=2.91e-04
Epoch  30/100  loss=1.0544  lr=2.65e-04
Epoch  40/100  loss=0.9745  lr=2.25e-04
Epoch  50/100  loss=0.9096  lr=1.76e-04
Epoch  60/100  loss=0.8726  lr=1.24e-04
Epoch  70/100  loss=0.8537  lr=7.50e-05
Epoch  80/100  loss=0.8420  lr=3.51e-05
Epoch  90/100  loss=0.8351  lr=9.05e-06
Epoch 100/100  loss=0.8256  lr=0.00e+00
Training complete. Saved to /home/svelag11/agentic/tgibs_v4_model.pt


In [195]:
# ── TF-GIBS v4 Inference + AUROC ─────────────────────────────────────────────
policy_text = (info.get('environment_info') or {}).get('policy', '')
tgibs_v4_model.eval()
all_consensus_tfv4 = {}

with tqdm(total=len(TASKS), desc='TF-GIBSv4 inference', unit='task') as pbar:
    for tid in TASKS:
        task     = tasks_map[tid]
        all_sims = sims_by_task.get(tid, [])
        for run in all_sims:
            try:
                sg = build_step_graph(run, info, task)
            except Exception:
                continue
            nodes = sg['nodes']
            agent_nodes_with_idx = [(i, n) for i, n in enumerate(nodes)
                                    if n['type'] == 'AgentStep']
            if not agent_nodes_with_idx: continue

            q_texts   = [n.get('op_text') or '' for _, n in agent_nodes_with_idx]
            user_msgs = [get_last_user_text(nodes, i) for i, _ in agent_nodes_with_idx]
            user_align   = [sim(qt, um) if um else 0.5 for qt, um in zip(q_texts, user_msgs)]
            policy_score = [sim(qt, policy_text) if policy_text else 0.5 for qt in q_texts]

            seq_texts, seq_types, agent_positions = [policy_text], [TYPE_POLICY], []
            for n in nodes:
                if n['type'] == 'UserTurn':
                    seq_texts.append(n.get('op_text') or '')
                    seq_types.append(TYPE_USER)
                elif n['type'] == 'AgentStep':
                    agent_positions.append(len(seq_texts))
                    seq_texts.append(n.get('op_text') or '')
                    seq_types.append(TYPE_AGENT)

            base_embs = torch.cat([policy_emb.unsqueeze(0), encode_texts(seq_texts[1:])], dim=0)
            agent_iter = iter(zip(user_align, policy_score))
            rows = []
            for pos, ttype in enumerate(seq_types):
                if ttype == TYPE_AGENT:
                    ua, ps = next(agent_iter)
                    rows.append(torch.cat([base_embs[pos], torch.tensor([ua, ps])]))
                else:
                    rows.append(torch.cat([base_embs[pos], ZERO_PAD]))
            seq_embs_v4 = torch.stack(rows).to(DEVICE)
            ttypes = torch.tensor(seq_types, dtype=torch.long).to(DEVICE)
            amask  = torch.zeros(len(seq_types), dtype=torch.bool).to(DEVICE)
            for pos in agent_positions:
                amask[pos] = True

            with torch.no_grad():
                p = tgibs_v4_model(seq_embs_v4, ttypes, amask)

            orig_indices = [i for i, n in enumerate(nodes) if n['type'] == 'AgentStep']
            all_consensus_tfv4[run['id']] = {
                orig_idx: {'m': float(p[pos])}
                for pos, orig_idx in enumerate(orig_indices)
            }
        pbar.update(1)

print(f'Runs scored: {len(all_consensus_tfv4)}')

y_true, s_tfv4 = [], []
for row in annots:
    sd = all_consensus_tfv4.get(row['run_id'], {}).get(int(row['step_idx']))
    if sd is None: continue
    y_true.append(1 if row['label'] in ('incorrect', 'tool_error') else 0)
    s_tfv4.append(1 - sd['m'])

print(f'Steps matched    : {len(y_true)}  (pos={sum(y_true)}, neg={len(y_true)-sum(y_true)})')
print()
print(f'AUROC TF-v4      : {roc_auc_score(y_true, s_tfv4):.4f}  <- full dialogue + NLI scalars')
print(f'--- known results ---')
print(f'NA-MCS           :  0.8101')
print(f'GCN-GIBS         :  0.7615')
print(f'TF-GIBS v2       :  0.6943  (NLI scalars, agent-only sequence)')
print(f'TF-GIBS v3       :  0.6635  (full dialogue, no NLI scalars)')


TF-GIBSv4 inference: 100%|██████████| 9/9 [00:03<00:00,  2.66task/s]

Runs scored: 180
Steps matched    : 2857  (pos=210, neg=2647)

AUROC TF-v4      : 0.7086  <- full dialogue + NLI scalars
--- known results ---
NA-MCS           :  0.8101
GCN-GIBS         :  0.7615
TF-GIBS v2       :  0.6943  (NLI scalars, agent-only sequence)
TF-GIBS v3       :  0.6635  (full dialogue, no NLI scalars)


In [196]:
# ── TF-v4 Expanded: build training data from 40 tasks ────────────────────────
TGIBS_V4_EXP_DATA_PATH = DRIVE_BASE + '/tgibs_v4_expanded_data.pkl'

if os.path.exists(TGIBS_V4_EXP_DATA_PATH):
    with open(TGIBS_V4_EXP_DATA_PATH, 'rb') as f:
        all_tgibs_v4_exp_data = pickle.load(f)
    print(f'Loaded cached TF-v4 expanded data: {len(all_tgibs_v4_exp_data)} runs')
else:
    all_tgibs_v4_exp_data = []
    for tid in tqdm(TRAIN_TASKS_EXPANDED, desc='Building TF-v4 expanded data'):
        all_tgibs_v4_exp_data.extend(build_tgibs_v4_data(tid))
    with open(TGIBS_V4_EXP_DATA_PATH, 'wb') as f:
        pickle.dump(all_tgibs_v4_exp_data, f)
    print(f'Built and cached: {len(all_tgibs_v4_exp_data)} runs')

print(f'Sequences: {len(all_tgibs_v4_exp_data)}')


Building TF-v4 expanded data: 100%|██████████| 40/40 [00:50<00:00,  1.27s/it]


Built and cached: 800 runs
Sequences: 800


In [197]:
# ── TF-v4 Expanded: train (AdamW + warmup + lam=2.0, 100 epochs) ─────────────
TGIBS_V4_EXP_MODEL_PATH = DRIVE_BASE + '/tgibs_v4_expanded_model.pt'

tgibs_v4_exp_model = TransformerGIBSv4().to(DEVICE)

if os.path.exists(TGIBS_V4_EXP_MODEL_PATH):
    tgibs_v4_exp_model.load_state_dict(torch.load(TGIBS_V4_EXP_MODEL_PATH, map_location=DEVICE))
    tgibs_v4_exp_model.eval()
    print(f'Loaded saved model from {TGIBS_V4_EXP_MODEL_PATH}')
else:
    TOTAL_EPOCHS, WARMUP_EPOCHS = 100, 10
    optimizer    = torch.optim.AdamW(tgibs_v4_exp_model.parameters(), lr=3e-4, weight_decay=0.01)
    warmup_sched = torch.optim.lr_scheduler.LambdaLR(optimizer,
                       lambda e: (e + 1) / WARMUP_EPOCHS if e < WARMUP_EPOCHS else 1.0)
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS - WARMUP_EPOCHS)
    scheduler    = torch.optim.lr_scheduler.SequentialLR(
                       optimizer, [warmup_sched, cosine_sched], milestones=[WARMUP_EPOCHS])
    tgibs_v4_exp_model.train()
    for epoch in range(1, TOTAL_EPOCHS + 1):
        total_loss = 0.0
        random.shuffle(all_tgibs_v4_exp_data)
        for seq_embs, token_types, agent_mask, soft_mask in all_tgibs_v4_exp_data:
            seq_embs    = seq_embs.to(DEVICE)
            token_types = token_types.to(DEVICE)
            agent_mask  = agent_mask.to(DEVICE)
            soft_mask   = soft_mask.to(DEVICE)
            optimizer.zero_grad()
            p    = tgibs_v4_exp_model(seq_embs, token_types, agent_mask)
            loss = ib_loss(p, soft_mask, lam=2.0)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(tgibs_v4_exp_model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        if epoch % 10 == 0:
            lr = scheduler.get_last_lr()[0]
            print(f'Epoch {epoch:3d}/{TOTAL_EPOCHS}  loss={total_loss/len(all_tgibs_v4_exp_data):.4f}  lr={lr:.2e}')
    torch.save(tgibs_v4_exp_model.state_dict(), TGIBS_V4_EXP_MODEL_PATH)
    print(f'Training complete. Saved to {TGIBS_V4_EXP_MODEL_PATH}')


Epoch  10/100  loss=1.0616  lr=3.00e-04
Epoch  20/100  loss=0.8658  lr=2.91e-04
Epoch  30/100  loss=0.7780  lr=2.65e-04
Epoch  40/100  loss=0.7461  lr=2.25e-04
Epoch  50/100  loss=0.6938  lr=1.76e-04
Epoch  60/100  loss=0.6665  lr=1.24e-04
Epoch  70/100  loss=0.6481  lr=7.50e-05
Epoch  80/100  loss=0.6292  lr=3.51e-05
Epoch  90/100  loss=0.6215  lr=9.05e-06
Epoch 100/100  loss=0.6209  lr=0.00e+00
Training complete. Saved to /home/svelag11/agentic/tgibs_v4_expanded_model.pt


In [198]:
# ── TF-v4 Expanded: Inference + AUROC ────────────────────────────────────────
policy_text = (info.get('environment_info') or {}).get('policy', '')
tgibs_v4_exp_model.eval()
all_consensus_tfv4_exp = {}

with tqdm(total=len(TASKS), desc='TF-v4-exp inference', unit='task') as pbar:
    for tid in TASKS:
        task     = tasks_map[tid]
        all_sims = sims_by_task.get(tid, [])
        for run in all_sims:
            try:
                sg = build_step_graph(run, info, task)
            except Exception:
                continue
            nodes = sg['nodes']
            agent_nodes_with_idx = [(i, n) for i, n in enumerate(nodes)
                                    if n['type'] == 'AgentStep']
            if not agent_nodes_with_idx: continue

            q_texts   = [n.get('op_text') or '' for _, n in agent_nodes_with_idx]
            user_msgs = [get_last_user_text(nodes, i) for i, _ in agent_nodes_with_idx]
            user_align   = [sim(qt, um) if um else 0.5 for qt, um in zip(q_texts, user_msgs)]
            policy_score = [sim(qt, policy_text) if policy_text else 0.5 for qt in q_texts]

            seq_texts, seq_types, agent_positions = [policy_text], [TYPE_POLICY], []
            for n in nodes:
                if n['type'] == 'UserTurn':
                    seq_texts.append(n.get('op_text') or '')
                    seq_types.append(TYPE_USER)
                elif n['type'] == 'AgentStep':
                    agent_positions.append(len(seq_texts))
                    seq_texts.append(n.get('op_text') or '')
                    seq_types.append(TYPE_AGENT)

            base_embs = torch.cat([policy_emb.unsqueeze(0), encode_texts(seq_texts[1:])], dim=0)
            agent_iter = iter(zip(user_align, policy_score))
            rows = []
            for pos, ttype in enumerate(seq_types):
                if ttype == TYPE_AGENT:
                    ua, ps = next(agent_iter)
                    rows.append(torch.cat([base_embs[pos], torch.tensor([ua, ps])]))
                else:
                    rows.append(torch.cat([base_embs[pos], ZERO_PAD]))
            seq_embs_v4 = torch.stack(rows).to(DEVICE)
            ttypes = torch.tensor(seq_types, dtype=torch.long).to(DEVICE)
            amask  = torch.zeros(len(seq_types), dtype=torch.bool).to(DEVICE)
            for pos in agent_positions:
                amask[pos] = True

            with torch.no_grad():
                p = tgibs_v4_exp_model(seq_embs_v4, ttypes, amask)

            orig_indices = [i for i, n in enumerate(nodes) if n['type'] == 'AgentStep']
            all_consensus_tfv4_exp[run['id']] = {
                orig_idx: {'m': float(p[pos])}
                for pos, orig_idx in enumerate(orig_indices)
            }
        pbar.update(1)

print(f'Runs scored: {len(all_consensus_tfv4_exp)}')

y_true, s = [], []
for row in annots:
    sd = all_consensus_tfv4_exp.get(row['run_id'], {}).get(int(row['step_idx']))
    if sd is None: continue
    y_true.append(1 if row['label'] in ('incorrect', 'tool_error') else 0)
    s.append(1 - sd['m'])

print(f'Steps matched       : {len(y_true)}  (pos={sum(y_true)}, neg={len(y_true)-sum(y_true)})')
print()
print(f'AUROC TF-v4 expanded: {roc_auc_score(y_true, s):.4f}  <- full dialogue + NLI + 40 tasks')
print(f'--- known results ---')
print(f'NA-MCS              :  0.8101')
print(f'GCN-GIBS            :  0.7615')
print(f'TF-GIBS v2          :  0.6943  (NLI scalars only, 15 tasks)')
print(f'TF-GIBS v3          :  0.6635  (full dialogue only, 15 tasks)')


TF-v4-exp inference: 100%|██████████| 9/9 [00:03<00:00,  2.65task/s]

Runs scored: 180
Steps matched       : 2857  (pos=210, neg=2647)

AUROC TF-v4 expanded: 0.6827  <- full dialogue + NLI + 40 tasks
--- known results ---
NA-MCS              :  0.8101
GCN-GIBS            :  0.7615
TF-GIBS v2          :  0.6943  (NLI scalars only, 15 tasks)
TF-GIBS v3          :  0.6635  (full dialogue only, 15 tasks)
